# FamilyBroker: этап 1, расчёт хеджей

Run All пересчитывает все результаты. Требуются Python 3 и numpy. Исторические котировки встроены; подключение к сети не требуется.

**Данные рынка реальные; ставки размещения, ГО, комиссии и коммерческие условия NDF — допущения.** Итог H1 — историческая реконструкция, не брокерский отчёт. Полный текст анализа и ответы на задания 1/1Б находятся в PDF.

Оценка сделана на закрытии 30.12.2025 с уже опубликованным курсом 78,2267 на 31.12. Входные размеры не используют будущую историю. Этап 2 выполняется после приглашения, в этот пакет не включён.

## Источники и ограничения
- [MOEX SiM6](https://iss.moex.com/iss/history/engines/futures/markets/forts/securities/SiM6.json?from=2025-12-30&till=2026-06-30), аналогичный запрос SiU6; страницы start=0,100,200.
- [ЦБ USD](https://www.cbr.ru/scripts/XML_dynamic.asp?date_req1=30/12/2025&date_req2=30/06/2026&VAL_NM_RQ=R01235).
- [Спецификация](https://iss.moex.com/iss/securities/SiM6.json): экспирация 18.06.2026; SiU6 17.09.2026.
- [Ключевая ставка](https://www.cbr.ru/rbr/dir_decisions/rsd_2025-12-19_20_01/): 16% на входе.

Результат в рублях W — сумма до конвертации, доллары U=W/[S(1+b)]. Эквивалент U×S₀ показывает валютную покупательную способность в рублях базы. Опционы не выбраны, их премии неприменимы. Налоги исключены. Ставка 14% не утверждается как историческая доходность. ГО фиксировано условно; риск его увеличения показан отдельно. ВМ за день используется как экономический денежный поток; до 23.03 не восстановлены два внутридневных клиринга, после — точное расписание T+1. Для оплаты в последний день добавлен однодневный мост 20% на ГО и последнюю положительную ВМ. Банковский NDF предполагает лимит без CSA, единый фиксинг и расчёт в дату платежа.

Все параметры находятся в следующей ячейке; изменение r, g, fee, basis пересчитывает модель и размер фьючерсного хеджа. NDF Q привязан к прогнозному доходу A; при иной фактической ставке нужно сохранять исходный Q, как сделано в отдельной чувствительности ниже.

In [1]:
import json, math, pathlib, datetime as dt, xml.etree.ElementTree as ET
import numpy as np
# Source observations embedded below
B=100_000_000.; S0=78.2267; r=.14; g=.15; fee=20.; basis=.005
start=dt.date(2025,12,30); end=dt.date(2026,6,30); roll=dt.date(2026,6,17)
em=dt.date(2026,6,18); eu=dt.date(2026,9,17)
days=(end-start).days; T=days/365
dates=[start+dt.timedelta(days=i) for i in range(days+1)]
raw={'SiM6': [{'BOARDID': 'RFUD', 'TRADEDATE': '2025-12-30', 'SECID': 'SiM6', 'OPEN': 82340.0, 'LOW': 81765.0, 'HIGH': 83215.0, 'CLOSE': 82890.0, 'OPENPOSITIONVALUE': 11477780160.0, 'VALUE': 2398003453.0, 'VOLUME': 28993, 'OPENPOSITION': 138460, 'SETTLEPRICE': 82896.0, 'SWAPRATE': 0.0, 'WAPRICE': 82726.0, 'CHANGE': 0.76, 'QTY': 4, 'NUMTRADES': 6442, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-05', 'SECID': 'SiM6', 'OPEN': 82878.0, 'LOW': 82761.0, 'HIGH': 84298.0, 'CLOSE': 84080.0, 'OPENPOSITIONVALUE': 11871927840.0, 'VALUE': 399758755.0, 'VOLUME': 4781, 'OPENPOSITION': 141198, 'SETTLEPRICE': 84080.0, 'SWAPRATE': 0.0, 'WAPRICE': 83628.0, 'CHANGE': 1.44, 'QTY': 1, 'NUMTRADES': 2480, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-06', 'SECID': 'SiM6', 'OPEN': 84098.0, 'LOW': 83289.0, 'HIGH': 84410.0, 'CLOSE': 83363.0, 'OPENPOSITIONVALUE': 11658229440.0, 'VALUE': 328484321.0, 'VOLUME': 3920, 'OPENPOSITION': 139854, 'SETTLEPRICE': 83360.0, 'SWAPRATE': 0.0, 'WAPRICE': 83792.0, 'CHANGE': -0.85, 'QTY': 1, 'NUMTRADES': 2677, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-08', 'SECID': 'SiM6', 'OPEN': 83365.0, 'LOW': 83186.0, 'HIGH': 84248.0, 'CLOSE': 83271.0, 'OPENPOSITIONVALUE': 11761836912.0, 'VALUE': 404435963.0, 'VOLUME': 4838, 'OPENPOSITION': 141246, 'SETTLEPRICE': 83272.0, 'SWAPRATE': 0.0, 'WAPRICE': 83598.0, 'CHANGE': -0.11, 'QTY': 1, 'NUMTRADES': 2479, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-09', 'SECID': 'SiM6', 'OPEN': 83346.0, 'LOW': 82497.0, 'HIGH': 83498.0, 'CLOSE': 82540.0, 'OPENPOSITIONVALUE': 11736284032.0, 'VALUE': 269681858.0, 'VOLUME': 3254, 'OPENPOSITION': 142208, 'SETTLEPRICE': 82529.0, 'SWAPRATE': 0.0, 'WAPRICE': 82866.0, 'CHANGE': -0.88, 'QTY': 2, 'NUMTRADES': 1662, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-12', 'SECID': 'SiM6', 'OPEN': 82568.0, 'LOW': 82274.0, 'HIGH': 83368.0, 'CLOSE': 82790.0, 'OPENPOSITIONVALUE': 11962672740.0, 'VALUE': 545173670.0, 'VOLUME': 6579, 'OPENPOSITION': 144468, 'SETTLEPRICE': 82805.0, 'SWAPRATE': 0.0, 'WAPRICE': 82875.0, 'CHANGE': 0.3, 'QTY': 1, 'NUMTRADES': 3048, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-13', 'SECID': 'SiM6', 'OPEN': 82773.0, 'LOW': 82573.0, 'HIGH': 83140.0, 'CLOSE': 82813.0, 'OPENPOSITIONVALUE': 11989213686.0, 'VALUE': 591701226.0, 'VOLUME': 7139, 'OPENPOSITION': 144834, 'SETTLEPRICE': 82779.0, 'SWAPRATE': 0.0, 'WAPRICE': 82882.0, 'CHANGE': 0.03, 'QTY': 1, 'NUMTRADES': 3375, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-14', 'SECID': 'SiM6', 'OPEN': 82828.0, 'LOW': 82100.0, 'HIGH': 83100.0, 'CLOSE': 82214.0, 'OPENPOSITIONVALUE': 12332858554.0, 'VALUE': 589871584.0, 'VOLUME': 7153, 'OPENPOSITION': 150022, 'SETTLEPRICE': 82207.0, 'SWAPRATE': 0.0, 'WAPRICE': 82461.0, 'CHANGE': -0.72, 'QTY': 5, 'NUMTRADES': 3203, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-15', 'SECID': 'SiM6', 'OPEN': 82170.0, 'LOW': 81685.0, 'HIGH': 82841.0, 'CLOSE': 81895.0, 'OPENPOSITIONVALUE': 13181948766.0, 'VALUE': 1143584383.0, 'VOLUME': 13866, 'OPENPOSITION': 160938, 'SETTLEPRICE': 81907.0, 'SWAPRATE': 0.0, 'WAPRICE': 82488.0, 'CHANGE': -0.39, 'QTY': 1, 'NUMTRADES': 4652, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-16', 'SECID': 'SiM6', 'OPEN': 81960.0, 'LOW': 81480.0, 'HIGH': 82198.0, 'CLOSE': 82159.0, 'OPENPOSITIONVALUE': 13292073756.0, 'VALUE': 505907464.0, 'VOLUME': 6184, 'OPENPOSITION': 161834, 'SETTLEPRICE': 82134.0, 'SWAPRATE': 0.0, 'WAPRICE': 81807.0, 'CHANGE': 0.32, 'QTY': 1, 'NUMTRADES': 3129, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-19', 'SECID': 'SiM6', 'OPEN': 82080.0, 'LOW': 81559.0, 'HIGH': 82209.0, 'CLOSE': 81700.0, 'OPENPOSITIONVALUE': 13983826222.0, 'VALUE': 752713502.0, 'VOLUME': 9206, 'OPENPOSITION': 171146, 'SETTLEPRICE': 81707.0, 'SWAPRATE': 0.0, 'WAPRICE': 81765.0, 'CHANGE': -0.56, 'QTY': 5, 'NUMTRADES': 3135, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-20', 'SECID': 'SiM6', 'OPEN': 81750.0, 'LOW': 81444.0, 'HIGH': 82159.0, 'CLOSE': 81701.0, 'OPENPOSITIONVALUE': 14268724060.0, 'VALUE': 924402165.0, 'VOLUME': 11296, 'OPENPOSITION': 174620, 'SETTLEPRICE': 81713.0, 'SWAPRATE': 0.0, 'WAPRICE': 81829.0, 'CHANGE': 0.0, 'QTY': 2, 'NUMTRADES': 4641, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-21', 'SECID': 'SiM6', 'OPEN': 81700.0, 'LOW': 80496.0, 'HIGH': 81918.0, 'CLOSE': 80539.0, 'OPENPOSITIONVALUE': 15187065810.0, 'VALUE': 1572104960.0, 'VOLUME': 19340, 'OPENPOSITION': 188610, 'SETTLEPRICE': 80521.0, 'SWAPRATE': 0.0, 'WAPRICE': 81271.0, 'CHANGE': -1.42, 'QTY': 1, 'NUMTRADES': 6492, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-22', 'SECID': 'SiM6', 'OPEN': 80556.0, 'LOW': 79300.0, 'HIGH': 80596.0, 'CLOSE': 79394.0, 'OPENPOSITIONVALUE': 16690991600.0, 'VALUE': 2787769744.0, 'VOLUME': 34962, 'OPENPOSITION': 210214, 'SETTLEPRICE': 79400.0, 'SWAPRATE': 0.0, 'WAPRICE': 79710.0, 'CHANGE': -1.42, 'QTY': 1, 'NUMTRADES': 12766, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-23', 'SECID': 'SiM6', 'OPEN': 79405.0, 'LOW': 79066.0, 'HIGH': 80150.0, 'CLOSE': 79491.0, 'OPENPOSITIONVALUE': 19366995000.0, 'VALUE': 2898819997.0, 'VOLUME': 36390, 'OPENPOSITION': 243610, 'SETTLEPRICE': 79500.0, 'SWAPRATE': 0.0, 'WAPRICE': 79662.0, 'CHANGE': 0.12, 'QTY': 1, 'NUMTRADES': 10751, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-26', 'SECID': 'SiM6', 'OPEN': 79513.0, 'LOW': 79031.0, 'HIGH': 80466.0, 'CLOSE': 80300.0, 'OPENPOSITIONVALUE': 24152577636.0, 'VALUE': 4185942597.0, 'VOLUME': 52443, 'OPENPOSITION': 300828, 'SETTLEPRICE': 80287.0, 'SWAPRATE': 0.0, 'WAPRICE': 79856.0, 'CHANGE': 1.02, 'QTY': 3, 'NUMTRADES': 12085, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-27', 'SECID': 'SiM6', 'OPEN': 80307.0, 'LOW': 79579.0, 'HIGH': 80609.0, 'CLOSE': 79620.0, 'OPENPOSITIONVALUE': 24805749420.0, 'VALUE': 1681485629.0, 'VOLUME': 21030, 'OPENPOSITION': 311540, 'SETTLEPRICE': 79623.0, 'SWAPRATE': 0.0, 'WAPRICE': 79964.0, 'CHANGE': -0.85, 'QTY': 1, 'NUMTRADES': 6736, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-28', 'SECID': 'SiM6', 'OPEN': 79639.0, 'LOW': 79184.0, 'HIGH': 80080.0, 'CLOSE': 80008.0, 'OPENPOSITIONVALUE': 30348370080.0, 'VALUE': 3864387916.0, 'VOLUME': 48518, 'OPENPOSITION': 379440, 'SETTLEPRICE': 79982.0, 'SWAPRATE': 0.0, 'WAPRICE': 79765.0, 'CHANGE': 0.49, 'QTY': 1, 'NUMTRADES': 9486, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-29', 'SECID': 'SiM6', 'OPEN': 79981.0, 'LOW': 78481.0, 'HIGH': 80035.0, 'CLOSE': 78695.0, 'OPENPOSITIONVALUE': 31129197924.0, 'VALUE': 2131833328.0, 'VOLUME': 26906, 'OPENPOSITION': 395628, 'SETTLEPRICE': 78683.0, 'SWAPRATE': 0.0, 'WAPRICE': 79207.0, 'CHANGE': -1.64, 'QTY': 2, 'NUMTRADES': 10869, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-30', 'SECID': 'SiM6', 'OPEN': 78701.0, 'LOW': 78630.0, 'HIGH': 79798.0, 'CLOSE': 79566.0, 'OPENPOSITIONVALUE': 32634001322.0, 'VALUE': 2182540661.0, 'VOLUME': 27514, 'OPENPOSITION': 410114, 'SETTLEPRICE': 79573.0, 'SWAPRATE': 0.0, 'WAPRICE': 79327.0, 'CHANGE': 1.11, 'QTY': 1, 'NUMTRADES': 9545, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-02', 'SECID': 'SiM6', 'OPEN': 79577.0, 'LOW': 79405.0, 'HIGH': 80945.0, 'CLOSE': 79935.0, 'OPENPOSITIONVALUE': 33250950336.0, 'VALUE': 1471153572.0, 'VOLUME': 18338, 'OPENPOSITION': 415824, 'SETTLEPRICE': 79964.0, 'SWAPRATE': 0.0, 'WAPRICE': 80229.0, 'CHANGE': 0.46, 'QTY': 5, 'NUMTRADES': 8325, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-03', 'SECID': 'SiM6', 'OPEN': 79950.0, 'LOW': 79873.0, 'HIGH': 80429.0, 'CLOSE': 80339.0, 'OPENPOSITIONVALUE': 34295639840.0, 'VALUE': 1413654669.0, 'VOLUME': 17638, 'OPENPOSITION': 426860, 'SETTLEPRICE': 80344.0, 'SWAPRATE': 0.0, 'WAPRICE': 80152.0, 'CHANGE': 0.51, 'QTY': 1, 'NUMTRADES': 7370, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-04', 'SECID': 'SiM6', 'OPEN': 80354.0, 'LOW': 79450.0, 'HIGH': 80636.0, 'CLOSE': 79590.0, 'OPENPOSITIONVALUE': 36152961600.0, 'VALUE': 2961759769.0, 'VOLUME': 37081, 'OPENPOSITION': 454240, 'SETTLEPRICE': 79590.0, 'SWAPRATE': 0.0, 'WAPRICE': 79874.0, 'CHANGE': -0.93, 'QTY': 2, 'NUMTRADES': 10789, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-05', 'SECID': 'SiM6', 'OPEN': 79669.0, 'LOW': 79090.0, 'HIGH': 80031.0, 'CLOSE': 79885.0, 'OPENPOSITIONVALUE': 39337966000.0, 'VALUE': 2511561232.0, 'VOLUME': 31532, 'OPENPOSITION': 492340, 'SETTLEPRICE': 79900.0, 'SWAPRATE': 0.0, 'WAPRICE': 79592.0, 'CHANGE': 0.37, 'QTY': 1, 'NUMTRADES': 7958, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-06', 'SECID': 'SiM6', 'OPEN': 79886.0, 'LOW': 79359.0, 'HIGH': 80500.0, 'CLOSE': 79871.0, 'OPENPOSITIONVALUE': 41673992606.0, 'VALUE': 2741423018.0, 'VOLUME': 34303, 'OPENPOSITION': 521714, 'SETTLEPRICE': 79879.0, 'SWAPRATE': 0.0, 'WAPRICE': 79919.0, 'CHANGE': -0.02, 'QTY': 1, 'NUMTRADES': 11442, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-09', 'SECID': 'SiM6', 'OPEN': 79914.0, 'LOW': 79622.0, 'HIGH': 80491.0, 'CLOSE': 80049.0, 'OPENPOSITIONVALUE': 46825692480.0, 'VALUE': 3843007263.0, 'VOLUME': 47947, 'OPENPOSITION': 585248, 'SETTLEPRICE': 80010.0, 'SWAPRATE': 0.0, 'WAPRICE': 80116.0, 'CHANGE': 0.22, 'QTY': 2, 'NUMTRADES': 7478, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-10', 'SECID': 'SiM6', 'OPEN': 80036.0, 'LOW': 79650.0, 'HIGH': 80136.0, 'CLOSE': 79900.0, 'OPENPOSITIONVALUE': 47341802144.0, 'VALUE': 1494411482.0, 'VOLUME': 18684, 'OPENPOSITION': 592528, 'SETTLEPRICE': 79898.0, 'SWAPRATE': 0.0, 'WAPRICE': 79977.0, 'CHANGE': -0.19, 'QTY': 1, 'NUMTRADES': 7098, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-11', 'SECID': 'SiM6', 'OPEN': 79967.0, 'LOW': 79369.0, 'HIGH': 80234.0, 'CLOSE': 79530.0, 'OPENPOSITIONVALUE': 51809659440.0, 'VALUE': 3658883022.0, 'VOLUME': 45964, 'OPENPOSITION': 651448, 'SETTLEPRICE': 79530.0, 'SWAPRATE': 0.0, 'WAPRICE': 79631.0, 'CHANGE': -0.46, 'QTY': 2, 'NUMTRADES': 25844, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-12', 'SECID': 'SiM6', 'OPEN': 79577.0, 'LOW': 79370.0, 'HIGH': 79857.0, 'CLOSE': 79526.0, 'OPENPOSITIONVALUE': 54602567040.0, 'VALUE': 3212113410.0, 'VOLUME': 40368, 'OPENPOSITION': 686652, 'SETTLEPRICE': 79520.0, 'SWAPRATE': 0.0, 'WAPRICE': 79573.0, 'CHANGE': -0.01, 'QTY': 1, 'NUMTRADES': 14771, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-13', 'SECID': 'SiM6', 'OPEN': 79585.0, 'LOW': 78582.0, 'HIGH': 79615.0, 'CLOSE': 78607.0, 'OPENPOSITIONVALUE': 60713531376.0, 'VALUE': 6508119533.0, 'VOLUME': 82260, 'OPENPOSITION': 772368, 'SETTLEPRICE': 78607.0, 'SWAPRATE': 0.0, 'WAPRICE': 79120.0, 'CHANGE': -1.16, 'QTY': 10, 'NUMTRADES': 44882, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-16', 'SECID': 'SiM6', 'OPEN': 78600.0, 'LOW': 78300.0, 'HIGH': 78927.0, 'CLOSE': 78851.0, 'OPENPOSITIONVALUE': 62470177856.0, 'VALUE': 2354321599.0, 'VOLUME': 29956, 'OPENPOSITION': 792256, 'SETTLEPRICE': 78851.0, 'SWAPRATE': 0.0, 'WAPRICE': 78591.0, 'CHANGE': 0.31, 'QTY': 1, 'NUMTRADES': 9798, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-17', 'SECID': 'SiM6', 'OPEN': 78855.0, 'LOW': 78086.0, 'HIGH': 78929.0, 'CLOSE': 78125.0, 'OPENPOSITIONVALUE': 64906817822.0, 'VALUE': 4517058332.0, 'VOLUME': 57546, 'OPENPOSITION': 830786, 'SETTLEPRICE': 78127.0, 'SWAPRATE': 0.0, 'WAPRICE': 78458.0, 'CHANGE': -0.92, 'QTY': 1, 'NUMTRADES': 14559, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-18', 'SECID': 'SiM6', 'OPEN': 78130.0, 'LOW': 77609.0, 'HIGH': 78851.0, 'CLOSE': 78577.0, 'OPENPOSITIONVALUE': 74172446838.0, 'VALUE': 6902407851.0, 'VOLUME': 88374, 'OPENPOSITION': 943922, 'SETTLEPRICE': 78579.0, 'SWAPRATE': 0.0, 'WAPRICE': 78090.0, 'CHANGE': 0.58, 'QTY': 2, 'NUMTRADES': 18123, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-19', 'SECID': 'SiM6', 'OPEN': 78535.0, 'LOW': 77729.0, 'HIGH': 78809.0, 'CLOSE': 77797.0, 'OPENPOSITIONVALUE': 90272571372.0, 'VALUE': 12604117504.0, 'VOLUME': 161080, 'OPENPOSITION': 1160286, 'SETTLEPRICE': 77802.0, 'SWAPRATE': 0.0, 'WAPRICE': 78159.0, 'CHANGE': -0.99, 'QTY': 6, 'NUMTRADES': 46484, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-20', 'SECID': 'SiM6', 'OPEN': 77802.0, 'LOW': 77757.0, 'HIGH': 78380.0, 'CLOSE': 78090.0, 'OPENPOSITIONVALUE': 118617616740.0, 'VALUE': 19923700810.0, 'VOLUME': 255714, 'OPENPOSITION': 1518986, 'SETTLEPRICE': 78090.0, 'SWAPRATE': 0.0, 'WAPRICE': 78089.0, 'CHANGE': 0.38, 'QTY': 2, 'NUMTRADES': 54768, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-24', 'SECID': 'SiM6', 'OPEN': 78082.0, 'LOW': 77312.0, 'HIGH': 78262.0, 'CLOSE': 77407.0, 'OPENPOSITIONVALUE': 145570192068.0, 'VALUE': 18140683185.0, 'VOLUME': 233439, 'OPENPOSITION': 1880436, 'SETTLEPRICE': 77413.0, 'SWAPRATE': 0.0, 'WAPRICE': 77686.0, 'CHANGE': -0.88, 'QTY': 1, 'NUMTRADES': 56144, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-25', 'SECID': 'SiM6', 'OPEN': 77403.0, 'LOW': 77207.0, 'HIGH': 79381.0, 'CLOSE': 78989.0, 'OPENPOSITIONVALUE': 167271160114.0, 'VALUE': 19016722565.0, 'VOLUME': 245060, 'OPENPOSITION': 2118134, 'SETTLEPRICE': 78971.0, 'SWAPRATE': 0.0, 'WAPRICE': 77677.0, 'CHANGE': 2.04, 'QTY': 25, 'NUMTRADES': 58212, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-26', 'SECID': 'SiM6', 'OPEN': 79030.0, 'LOW': 78218.0, 'HIGH': 79352.0, 'CLOSE': 78244.0, 'OPENPOSITIONVALUE': 176587170860.0, 'VALUE': 12785373037.0, 'VOLUME': 162745, 'OPENPOSITION': 2257282, 'SETTLEPRICE': 78230.0, 'SWAPRATE': 0.0, 'WAPRICE': 78522.0, 'CHANGE': -0.94, 'QTY': 1, 'NUMTRADES': 53842, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-27', 'SECID': 'SiM6', 'OPEN': 78275.0, 'LOW': 78109.0, 'HIGH': 78792.0, 'CLOSE': 78150.0, 'OPENPOSITIONVALUE': 206804824356.0, 'VALUE': 28370344518.0, 'VOLUME': 361424, 'OPENPOSITION': 2646492, 'SETTLEPRICE': 78143.0, 'SWAPRATE': 0.0, 'WAPRICE': 78529.0, 'CHANGE': -0.12, 'QTY': 1, 'NUMTRADES': 78547, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-02', 'SECID': 'SiM6', 'OPEN': 78125.0, 'LOW': 77058.0, 'HIGH': 79000.0, 'CLOSE': 78920.0, 'OPENPOSITIONVALUE': 234574493904.0, 'VALUE': 28513225197.0, 'VOLUME': 363408, 'OPENPOSITION': 2972232, 'SETTLEPRICE': 78922.0, 'SWAPRATE': 0.0, 'WAPRICE': 78453.0, 'CHANGE': 0.98, 'QTY': 1, 'NUMTRADES': 75666, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-03', 'SECID': 'SiM6', 'OPEN': 78885.0, 'LOW': 78489.0, 'HIGH': 79000.0, 'CLOSE': 78780.0, 'OPENPOSITIONVALUE': 251233215744.0, 'VALUE': 21344607453.0, 'VOLUME': 270999, 'OPENPOSITION': 3189696, 'SETTLEPRICE': 78764.0, 'SWAPRATE': 0.0, 'WAPRICE': 78762.0, 'CHANGE': -0.18, 'QTY': 1, 'NUMTRADES': 93490, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-04', 'SECID': 'SiM6', 'OPEN': 78774.0, 'LOW': 78500.0, 'HIGH': 79891.0, 'CLOSE': 79253.0, 'OPENPOSITIONVALUE': 284117397200.0, 'VALUE': 28844607680.0, 'VOLUME': 364610, 'OPENPOSITION': 3585530, 'SETTLEPRICE': 79240.0, 'SWAPRATE': 0.0, 'WAPRICE': 79161.0, 'CHANGE': 0.6, 'QTY': 14, 'NUMTRADES': 93408, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-05', 'SECID': 'SiM6', 'OPEN': 79291.0, 'LOW': 79200.0, 'HIGH': 80453.0, 'CLOSE': 80409.0, 'OPENPOSITIONVALUE': 319668693768.0, 'VALUE': 31216240578.0, 'VOLUME': 390765, 'OPENPOSITION': 3975336, 'SETTLEPRICE': 80413.0, 'SWAPRATE': 0.0, 'WAPRICE': 79916.0, 'CHANGE': 1.46, 'QTY': 6, 'NUMTRADES': 103460, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-06', 'SECID': 'SiM6', 'OPEN': 80408.0, 'LOW': 79516.0, 'HIGH': 80670.0, 'CLOSE': 79560.0, 'OPENPOSITIONVALUE': 361588040190.0, 'VALUE': 45110790399.0, 'VOLUME': 562828, 'OPENPOSITION': 4542278, 'SETTLEPRICE': 79605.0, 'SWAPRATE': 0.0, 'WAPRICE': 80130.0, 'CHANGE': -1.06, 'QTY': 4, 'NUMTRADES': 129890, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-09', 'SECID': 'SiM6', 'OPEN': 79577.0, 'LOW': 78603.0, 'HIGH': 79850.0, 'CLOSE': 78922.0, 'OPENPOSITIONVALUE': 360641738050.0, 'VALUE': 8973835410.0, 'VOLUME': 113364, 'OPENPOSITION': 4568150, 'SETTLEPRICE': 78947.0, 'SWAPRATE': 0.0, 'WAPRICE': 79152.0, 'CHANGE': -0.8, 'QTY': 2, 'NUMTRADES': 36770, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-10', 'SECID': 'SiM6', 'OPEN': 78941.0, 'LOW': 78850.0, 'HIGH': 79802.0, 'CLOSE': 79422.0, 'OPENPOSITIONVALUE': 462204957600.0, 'VALUE': 81939043384.0, 'VOLUME': 1030356, 'OPENPOSITION': 5818290, 'SETTLEPRICE': 79440.0, 'SWAPRATE': 0.0, 'WAPRICE': 79549.0, 'CHANGE': 0.63, 'QTY': 1, 'NUMTRADES': 173178, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-11', 'SECID': 'SiM6', 'OPEN': 79490.0, 'LOW': 79200.0, 'HIGH': 79857.0, 'CLOSE': 79656.0, 'OPENPOSITIONVALUE': 528816801760.0, 'VALUE': 54406007721.0, 'VOLUME': 684495, 'OPENPOSITION': 6638090, 'SETTLEPRICE': 79664.0, 'SWAPRATE': 0.0, 'WAPRICE': 79514.0, 'CHANGE': 0.29, 'QTY': 1, 'NUMTRADES': 142713, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-12', 'SECID': 'SiM6', 'OPEN': 79680.0, 'LOW': 79096.0, 'HIGH': 80215.0, 'CLOSE': 80050.0, 'OPENPOSITIONVALUE': 580367303000.0, 'VALUE': 49907537085.0, 'VOLUME': 627926, 'OPENPOSITION': 7250060, 'SETTLEPRICE': 80050.0, 'SWAPRATE': 0.0, 'WAPRICE': 79485.0, 'CHANGE': 0.5, 'QTY': 1, 'NUMTRADES': 134365, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-13', 'SECID': 'SiM6', 'OPEN': 80001.0, 'LOW': 80000.0, 'HIGH': 80783.0, 'CLOSE': 80595.0, 'OPENPOSITIONVALUE': 632735435304.0, 'VALUE': 60550200326.0, 'VOLUME': 752719, 'OPENPOSITION': 7849926, 'SETTLEPRICE': 80604.0, 'SWAPRATE': 0.0, 'WAPRICE': 80435.0, 'CHANGE': 0.68, 'QTY': 3, 'NUMTRADES': 142955, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-16', 'SECID': 'SiM6', 'OPEN': 80620.0, 'LOW': 80315.0, 'HIGH': 81631.0, 'CLOSE': 81568.0, 'OPENPOSITIONVALUE': 704105579840.0, 'VALUE': 78290498125.0, 'VOLUME': 965577, 'OPENPOSITION': 8632130, 'SETTLEPRICE': 81568.0, 'SWAPRATE': 0.0, 'WAPRICE': 81097.0, 'CHANGE': 1.21, 'QTY': 1, 'NUMTRADES': 202987, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-17', 'SECID': 'SiM6', 'OPEN': 81558.0, 'LOW': 81354.0, 'HIGH': 82733.0, 'CLOSE': 82719.0, 'OPENPOSITIONVALUE': 821349428152.0, 'VALUE': 121201510172.0, 'VOLUME': 1478052, 'OPENPOSITION': 9933836, 'SETTLEPRICE': 82682.0, 'SWAPRATE': 0.0, 'WAPRICE': 82058.0, 'CHANGE': 1.41, 'QTY': 5, 'NUMTRADES': 282572, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-18', 'SECID': 'SiM6', 'OPEN': 82725.0, 'LOW': 82159.0, 'HIGH': 83315.0, 'CLOSE': 83146.0, 'OPENPOSITIONVALUE': 922581283184.0, 'VALUE': 150262701227.0, 'VOLUME': 1812862, 'OPENPOSITION': 11099924, 'SETTLEPRICE': 83116.0, 'SWAPRATE': 0.0, 'WAPRICE': 82915.0, 'CHANGE': 0.52, 'QTY': 1, 'NUMTRADES': 395750, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-19', 'SECID': 'SiM6', 'OPEN': 83110.0, 'LOW': 82139.0, 'HIGH': 84525.0, 'CLOSE': 83494.0, 'OPENPOSITIONVALUE': 990904391200.0, 'VALUE': 191557184634.0, 'VOLUME': 2287666, 'OPENPOSITION': 11866550, 'SETTLEPRICE': 83504.0, 'SWAPRATE': 0.0, 'WAPRICE': 83735.0, 'CHANGE': 0.42, 'QTY': 1, 'NUMTRADES': 438647, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-20', 'SECID': 'SiM6', 'OPEN': 83472.0, 'LOW': 82177.0, 'HIGH': 84000.0, 'CLOSE': 82929.0, 'OPENPOSITIONVALUE': 961834466544.0, 'VALUE': 201064007378.0, 'VOLUME': 2415693, 'OPENPOSITION': 11601786, 'SETTLEPRICE': 82904.0, 'SWAPRATE': 0.0, 'WAPRICE': 83227.0, 'CHANGE': -0.68, 'QTY': 2, 'NUMTRADES': 477265, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-23', 'SECID': 'SiM6', 'OPEN': 83001.0, 'LOW': 81321.0, 'HIGH': 83483.0, 'CLOSE': 81400.0, 'OPENPOSITIONVALUE': 895847232280.0, 'VALUE': 174733265683.0, 'VOLUME': 2119469, 'OPENPOSITION': 10873252, 'SETTLEPRICE': 82390.0, 'SWAPRATE': 0.0, 'WAPRICE': 82442.0, 'CHANGE': -1.84, 'QTY': 7, 'NUMTRADES': 416238, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-24', 'SECID': 'SiM6', 'OPEN': 81400.0, 'LOW': 80088.0, 'HIGH': 81600.0, 'CLOSE': 80361.0, 'OPENPOSITIONVALUE': 887838782622.0, 'VALUE': 116954874241.0, 'VOLUME': 1448421, 'OPENPOSITION': 11074866, 'SETTLEPRICE': 80167.0, 'SWAPRATE': 0.0, 'WAPRICE': 80751.0, 'CHANGE': -1.28, 'QTY': 4, 'NUMTRADES': 275177, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-25', 'SECID': 'SiM6', 'OPEN': 80385.0, 'LOW': 80217.0, 'HIGH': 81689.0, 'CLOSE': 81373.0, 'OPENPOSITIONVALUE': 888671893614.0, 'VALUE': 85280336997.0, 'VOLUME': 1054682, 'OPENPOSITION': 10919894, 'SETTLEPRICE': 81381.0, 'SWAPRATE': 0.0, 'WAPRICE': 80862.0, 'CHANGE': 1.26, 'QTY': 1, 'NUMTRADES': 273738, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-26', 'SECID': 'SiM6', 'OPEN': 81415.0, 'LOW': 80617.0, 'HIGH': 82378.0, 'CLOSE': 80880.0, 'OPENPOSITIONVALUE': 860737682868.0, 'VALUE': 172853235988.0, 'VOLUME': 2117329, 'OPENPOSITION': 10668274, 'SETTLEPRICE': 80682.0, 'SWAPRATE': 0.0, 'WAPRICE': 81639.0, 'CHANGE': -0.61, 'QTY': 1, 'NUMTRADES': 423790, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-27', 'SECID': 'SiM6', 'OPEN': 80888.0, 'LOW': 80419.0, 'HIGH': 81965.0, 'CLOSE': 81732.0, 'OPENPOSITIONVALUE': 862394928776.0, 'VALUE': 96653071316.0, 'VOLUME': 1189529, 'OPENPOSITION': 10532296, 'SETTLEPRICE': 81881.0, 'SWAPRATE': 0.0, 'WAPRICE': 81254.0, 'CHANGE': 1.05, 'QTY': 9, 'NUMTRADES': 273245, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-30', 'SECID': 'SiM6', 'OPEN': 81676.0, 'LOW': 81157.0, 'HIGH': 82419.0, 'CLOSE': 81773.0, 'OPENPOSITIONVALUE': 862331063688.0, 'VALUE': 91990108776.0, 'VOLUME': 1125508, 'OPENPOSITION': 10495242, 'SETTLEPRICE': 82164.0, 'SWAPRATE': 0.0, 'WAPRICE': 81733.0, 'CHANGE': 0.05, 'QTY': 4, 'NUMTRADES': 248364, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-31', 'SECID': 'SiM6', 'OPEN': 81740.0, 'LOW': 81577.0, 'HIGH': 82793.0, 'CLOSE': 82356.0, 'OPENPOSITIONVALUE': 858822785164.0, 'VALUE': 109718520546.0, 'VOLUME': 1335147, 'OPENPOSITION': 10443646, 'SETTLEPRICE': 82234.0, 'SWAPRATE': 0.0, 'WAPRICE': 82179.0, 'CHANGE': 0.71, 'QTY': 1, 'NUMTRADES': 272851, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-01', 'SECID': 'SiM6', 'OPEN': 82330.0, 'LOW': 80750.0, 'HIGH': 82330.0, 'CLOSE': 81084.0, 'OPENPOSITIONVALUE': 843792778570.0, 'VALUE': 96393039148.0, 'VOLUME': 1186276, 'OPENPOSITION': 10421698, 'SETTLEPRICE': 80965.0, 'SWAPRATE': 0.0, 'WAPRICE': 81250.0, 'CHANGE': -1.55, 'QTY': 15, 'NUMTRADES': 261998, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-02', 'SECID': 'SiM6', 'OPEN': 81084.0, 'LOW': 80609.0, 'HIGH': 81390.0, 'CLOSE': 80897.0, 'OPENPOSITIONVALUE': 835981111504.0, 'VALUE': 81137991527.0, 'VOLUME': 1002026, 'OPENPOSITION': 10335556, 'SETTLEPRICE': 80884.0, 'SWAPRATE': 0.0, 'WAPRICE': 80974.0, 'CHANGE': -0.23, 'QTY': 1, 'NUMTRADES': 214804, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-03', 'SECID': 'SiM6', 'OPEN': 80896.0, 'LOW': 79850.0, 'HIGH': 80896.0, 'CLOSE': 79958.0, 'OPENPOSITIONVALUE': 823127794334.0, 'VALUE': 67067626861.0, 'VOLUME': 834549, 'OPENPOSITION': 10293086, 'SETTLEPRICE': 79969.0, 'SWAPRATE': 0.0, 'WAPRICE': 80360.0, 'CHANGE': -1.16, 'QTY': 8, 'NUMTRADES': 191730, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-06', 'SECID': 'SiM6', 'OPEN': 79890.0, 'LOW': 79240.0, 'HIGH': 80090.0, 'CLOSE': 79779.0, 'OPENPOSITIONVALUE': 826827404142.0, 'VALUE': 73124421917.0, 'VOLUME': 917456, 'OPENPOSITION': 10375678, 'SETTLEPRICE': 79689.0, 'SWAPRATE': 0.0, 'WAPRICE': 79708.0, 'CHANGE': -0.22, 'QTY': 1, 'NUMTRADES': 203271, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-07', 'SECID': 'SiM6', 'OPEN': 79749.0, 'LOW': 79191.0, 'HIGH': 80137.0, 'CLOSE': 79420.0, 'OPENPOSITIONVALUE': 794155334016.0, 'VALUE': 89607361110.0, 'VOLUME': 1125291, 'OPENPOSITION': 9998934, 'SETTLEPRICE': 79424.0, 'SWAPRATE': 0.0, 'WAPRICE': 79637.0, 'CHANGE': -0.45, 'QTY': 3, 'NUMTRADES': 233661, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-08', 'SECID': 'SiM6', 'OPEN': 79510.0, 'LOW': 79030.0, 'HIGH': 79798.0, 'CLOSE': 79342.0, 'OPENPOSITIONVALUE': 799893881280.0, 'VALUE': 77993015182.0, 'VOLUME': 982939, 'OPENPOSITION': 10067130, 'SETTLEPRICE': 79456.0, 'SWAPRATE': 0.0, 'WAPRICE': 79346.0, 'CHANGE': -0.1, 'QTY': 1, 'NUMTRADES': 207533, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-09', 'SECID': 'SiM6', 'OPEN': 79342.0, 'LOW': 78250.0, 'HIGH': 79400.0, 'CLOSE': 78265.0, 'OPENPOSITIONVALUE': 789229372544.0, 'VALUE': 79313374376.0, 'VOLUME': 1005989, 'OPENPOSITION': 10061696, 'SETTLEPRICE': 78439.0, 'SWAPRATE': 0.0, 'WAPRICE': 78835.0, 'CHANGE': -1.36, 'QTY': 5, 'NUMTRADES': 207490, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-10', 'SECID': 'SiM6', 'OPEN': 78276.0, 'LOW': 77420.0, 'HIGH': 78409.0, 'CLOSE': 77709.0, 'OPENPOSITIONVALUE': 773666314746.0, 'VALUE': 91166806685.0, 'VOLUME': 1171228, 'OPENPOSITION': 9955686, 'SETTLEPRICE': 77711.0, 'SWAPRATE': 0.0, 'WAPRICE': 77830.0, 'CHANGE': -0.71, 'QTY': 9, 'NUMTRADES': 234747, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-13', 'SECID': 'SiM6', 'OPEN': 77470.0, 'LOW': 76773.0, 'HIGH': 77521.0, 'CLOSE': 77060.0, 'OPENPOSITIONVALUE': 758554868528.0, 'VALUE': 113079853392.0, 'VOLUME': 1465568, 'OPENPOSITION': 9811352, 'SETTLEPRICE': 77314.0, 'SWAPRATE': 0.0, 'WAPRICE': 77156.0, 'CHANGE': -0.84, 'QTY': 2, 'NUMTRADES': 257673, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-14', 'SECID': 'SiM6', 'OPEN': 77220.0, 'LOW': 75776.0, 'HIGH': 77274.0, 'CLOSE': 75859.0, 'OPENPOSITIONVALUE': 759977262396.0, 'VALUE': 105959193481.0, 'VOLUME': 1386516, 'OPENPOSITION': 10007338, 'SETTLEPRICE': 75942.0, 'SWAPRATE': 0.0, 'WAPRICE': 76418.0, 'CHANGE': -1.56, 'QTY': 1, 'NUMTRADES': 236811, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-15', 'SECID': 'SiM6', 'OPEN': 75900.0, 'LOW': 75573.0, 'HIGH': 76968.0, 'CLOSE': 76581.0, 'OPENPOSITIONVALUE': 762588914340.0, 'VALUE': 94993808091.0, 'VOLUME': 1245430, 'OPENPOSITION': 9953130, 'SETTLEPRICE': 76618.0, 'SWAPRATE': 0.0, 'WAPRICE': 76287.0, 'CHANGE': 0.95, 'QTY': 4, 'NUMTRADES': 257052, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-16', 'SECID': 'SiM6', 'OPEN': 76551.0, 'LOW': 76185.0, 'HIGH': 77772.0, 'CLOSE': 77035.0, 'OPENPOSITIONVALUE': 761087157096.0, 'VALUE': 99565910706.0, 'VOLUME': 1289875, 'OPENPOSITION': 9895944, 'SETTLEPRICE': 76909.0, 'SWAPRATE': 0.0, 'WAPRICE': 77195.0, 'CHANGE': 0.59, 'QTY': 3, 'NUMTRADES': 273740, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-17', 'SECID': 'SiM6', 'OPEN': 77050.0, 'LOW': 76500.0, 'HIGH': 77449.0, 'CLOSE': 76852.0, 'OPENPOSITIONVALUE': 764610765120.0, 'VALUE': 78412861234.0, 'VOLUME': 1019499, 'OPENPOSITION': 9967680, 'SETTLEPRICE': 76709.0, 'SWAPRATE': 0.0, 'WAPRICE': 76913.0, 'CHANGE': -0.24, 'QTY': 1, 'NUMTRADES': 196598, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-20', 'SECID': 'SiM6', 'OPEN': 76700.0, 'LOW': 75878.0, 'HIGH': 76770.0, 'CLOSE': 75987.0, 'OPENPOSITIONVALUE': 738596127518.0, 'VALUE': 91939360468.0, 'VOLUME': 1205436, 'OPENPOSITION': 9694262, 'SETTLEPRICE': 76189.0, 'SWAPRATE': 0.0, 'WAPRICE': 76266.0, 'CHANGE': -1.13, 'QTY': 5, 'NUMTRADES': 213294, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-21', 'SECID': 'SiM6', 'OPEN': 75994.0, 'LOW': 75211.0, 'HIGH': 76089.0, 'CLOSE': 75489.0, 'OPENPOSITIONVALUE': 762631207176.0, 'VALUE': 110519110339.0, 'VOLUME': 1462885, 'OPENPOSITION': 10111656, 'SETTLEPRICE': 75421.0, 'SWAPRATE': 0.0, 'WAPRICE': 75545.0, 'CHANGE': -0.66, 'QTY': 4, 'NUMTRADES': 276925, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-22', 'SECID': 'SiM6', 'OPEN': 75489.0, 'LOW': 75350.0, 'HIGH': 76230.0, 'CLOSE': 75880.0, 'OPENPOSITIONVALUE': 738171685540.0, 'VALUE': 112062278347.0, 'VOLUME': 1476887, 'OPENPOSITION': 9745870, 'SETTLEPRICE': 75742.0, 'SWAPRATE': 0.0, 'WAPRICE': 75888.0, 'CHANGE': 0.52, 'QTY': 4, 'NUMTRADES': 274473, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-23', 'SECID': 'SiM6', 'OPEN': 75860.0, 'LOW': 75510.0, 'HIGH': 76970.0, 'CLOSE': 76811.0, 'OPENPOSITIONVALUE': 767622537864.0, 'VALUE': 141198043521.0, 'VOLUME': 1854610, 'OPENPOSITION': 10010988, 'SETTLEPRICE': 76678.0, 'SWAPRATE': 0.0, 'WAPRICE': 76138.0, 'CHANGE': 1.23, 'QTY': 1, 'NUMTRADES': 332258, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-24', 'SECID': 'SiM6', 'OPEN': 76850.0, 'LOW': 75810.0, 'HIGH': 77174.0, 'CLOSE': 75815.0, 'OPENPOSITIONVALUE': 789042548288.0, 'VALUE': 115277552333.0, 'VOLUME': 1509973, 'OPENPOSITION': 10399654, 'SETTLEPRICE': 75872.0, 'SWAPRATE': 0.0, 'WAPRICE': 76339.0, 'CHANGE': -1.3, 'QTY': 1, 'NUMTRADES': 267271, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-27', 'SECID': 'SiM6', 'OPEN': 75839.0, 'LOW': 75330.0, 'HIGH': 75888.0, 'CLOSE': 75476.0, 'OPENPOSITIONVALUE': 804504212240.0, 'VALUE': 66653923967.0, 'VOLUME': 881786, 'OPENPOSITION': 10657520, 'SETTLEPRICE': 75487.0, 'SWAPRATE': 0.0, 'WAPRICE': 75585.0, 'CHANGE': -0.45, 'QTY': 4, 'NUMTRADES': 170961, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-28', 'SECID': 'SiM6', 'OPEN': 75478.0, 'LOW': 75054.0, 'HIGH': 76379.0, 'CLOSE': 76146.0, 'OPENPOSITIONVALUE': 798030940386.0, 'VALUE': 117555453435.0, 'VOLUME': 1549480, 'OPENPOSITION': 10475734, 'SETTLEPRICE': 76179.0, 'SWAPRATE': 0.0, 'WAPRICE': 75872.0, 'CHANGE': 0.89, 'QTY': 5, 'NUMTRADES': 243492, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-29', 'SECID': 'SiM6', 'OPEN': 76166.0, 'LOW': 75340.0, 'HIGH': 76430.0, 'CLOSE': 75698.0, 'OPENPOSITIONVALUE': 809239682460.0, 'VALUE': 87159241648.0, 'VOLUME': 1151195, 'OPENPOSITION': 10708478, 'SETTLEPRICE': 75570.0, 'SWAPRATE': 0.0, 'WAPRICE': 75708.0, 'CHANGE': -0.59, 'QTY': 1, 'NUMTRADES': 221772, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-30', 'SECID': 'SiM6', 'OPEN': 75662.0, 'LOW': 75503.0, 'HIGH': 76232.0, 'CLOSE': 76025.0, 'OPENPOSITIONVALUE': 797530339046.0, 'VALUE': 76271047530.0, 'VOLUME': 1005550, 'OPENPOSITION': 10497826, 'SETTLEPRICE': 75971.0, 'SWAPRATE': 0.0, 'WAPRICE': 75856.0, 'CHANGE': 0.43, 'QTY': 38, 'NUMTRADES': 206590, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-04', 'SECID': 'SiM6', 'OPEN': 76010.0, 'LOW': 75750.0, 'HIGH': 76799.0, 'CLOSE': 76485.0, 'OPENPOSITIONVALUE': 800152701560.0, 'VALUE': 66719970373.0, 'VOLUME': 873973, 'OPENPOSITION': 10469092, 'SETTLEPRICE': 76430.0, 'SWAPRATE': 0.0, 'WAPRICE': 76347.0, 'CHANGE': 0.61, 'QTY': 2, 'NUMTRADES': 175968, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-05', 'SECID': 'SiM6', 'OPEN': 76498.0, 'LOW': 76218.0, 'HIGH': 76891.0, 'CLOSE': 76725.0, 'OPENPOSITIONVALUE': 803715063780.0, 'VALUE': 56014421801.0, 'VOLUME': 731946, 'OPENPOSITION': 10477318, 'SETTLEPRICE': 76710.0, 'SWAPRATE': 0.0, 'WAPRICE': 76537.0, 'CHANGE': 0.31, 'QTY': 1, 'NUMTRADES': 140048, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-06', 'SECID': 'SiM6', 'OPEN': 76700.0, 'LOW': 74997.0, 'HIGH': 76888.0, 'CLOSE': 75365.0, 'OPENPOSITIONVALUE': 811018773864.0, 'VALUE': 123127238762.0, 'VOLUME': 1625623, 'OPENPOSITION': 10758072, 'SETTLEPRICE': 75387.0, 'SWAPRATE': 0.0, 'WAPRICE': 75738.0, 'CHANGE': -1.77, 'QTY': 4, 'NUMTRADES': 309546, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-07', 'SECID': 'SiM6', 'OPEN': 75329.0, 'LOW': 75131.0, 'HIGH': 75777.0, 'CLOSE': 75490.0, 'OPENPOSITIONVALUE': 811840310512.0, 'VALUE': 57089485766.0, 'VOLUME': 756162, 'OPENPOSITION': 10755416, 'SETTLEPRICE': 75482.0, 'SWAPRATE': 0.0, 'WAPRICE': 75501.0, 'CHANGE': 0.17, 'QTY': 1, 'NUMTRADES': 180506, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-08', 'SECID': 'SiM6', 'OPEN': 75490.0, 'LOW': 74785.0, 'HIGH': 75546.0, 'CLOSE': 75015.0, 'OPENPOSITIONVALUE': 808564840534.0, 'VALUE': 44970984585.0, 'VOLUME': 598737, 'OPENPOSITION': 10763786, 'SETTLEPRICE': 75119.0, 'SWAPRATE': 0.0, 'WAPRICE': 75107.0, 'CHANGE': -0.63, 'QTY': 1, 'NUMTRADES': 136039, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-11', 'SECID': 'SiM6', 'OPEN': 74958.0, 'LOW': 74123.0, 'HIGH': 75134.0, 'CLOSE': 74300.0, 'OPENPOSITIONVALUE': 800349432552.0, 'VALUE': 30850273921.0, 'VOLUME': 414099, 'OPENPOSITION': 10786962, 'SETTLEPRICE': 74196.0, 'SWAPRATE': 0.0, 'WAPRICE': 74499.0, 'CHANGE': -0.95, 'QTY': 5, 'NUMTRADES': 100919, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-12', 'SECID': 'SiM6', 'OPEN': 74226.0, 'LOW': 74150.0, 'HIGH': 74996.0, 'CLOSE': 74554.0, 'OPENPOSITIONVALUE': 787522993740.0, 'VALUE': 69493062409.0, 'VOLUME': 931054, 'OPENPOSITION': 10558590, 'SETTLEPRICE': 74586.0, 'SWAPRATE': 0.0, 'WAPRICE': 74639.0, 'CHANGE': 0.34, 'QTY': 1, 'NUMTRADES': 157208, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-13', 'SECID': 'SiM6', 'OPEN': 74521.0, 'LOW': 73811.0, 'HIGH': 74573.0, 'CLOSE': 74303.0, 'OPENPOSITIONVALUE': 759164145882.0, 'VALUE': 93225691930.0, 'VOLUME': 1255747, 'OPENPOSITION': 10221542, 'SETTLEPRICE': 74271.0, 'SWAPRATE': 0.0, 'WAPRICE': 74227.0, 'CHANGE': -0.34, 'QTY': 11, 'NUMTRADES': 204828, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-14', 'SECID': 'SiM6', 'OPEN': 74260.0, 'LOW': 73709.0, 'HIGH': 74376.0, 'CLOSE': 73940.0, 'OPENPOSITIONVALUE': 757192279050.0, 'VALUE': 78437412489.0, 'VOLUME': 1059917, 'OPENPOSITION': 10237170, 'SETTLEPRICE': 73965.0, 'SWAPRATE': 0.0, 'WAPRICE': 74000.0, 'CHANGE': -0.49, 'QTY': 1, 'NUMTRADES': 194733, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-15', 'SECID': 'SiM6', 'OPEN': 74006.0, 'LOW': 73555.0, 'HIGH': 74106.0, 'CLOSE': 73654.0, 'OPENPOSITIONVALUE': 745353329028.0, 'VALUE': 55550083368.0, 'VOLUME': 752754, 'OPENPOSITION': 10114852, 'SETTLEPRICE': 73689.0, 'SWAPRATE': 0.0, 'WAPRICE': 73789.0, 'CHANGE': -0.39, 'QTY': 4, 'NUMTRADES': 134541, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-18', 'SECID': 'SiM6', 'OPEN': 73700.0, 'LOW': 72590.0, 'HIGH': 73775.0, 'CLOSE': 72592.0, 'OPENPOSITIONVALUE': 737322853470.0, 'VALUE': 62356664765.0, 'VOLUME': 852929, 'OPENPOSITION': 10105158, 'SETTLEPRICE': 72965.0, 'SWAPRATE': 0.0, 'WAPRICE': 73083.0, 'CHANGE': -1.44, 'QTY': 1, 'NUMTRADES': 157232, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-19', 'SECID': 'SiM6', 'OPEN': 72592.0, 'LOW': 70911.0, 'HIGH': 72717.0, 'CLOSE': 70922.0, 'OPENPOSITIONVALUE': 695333668952.0, 'VALUE': 101425281719.0, 'VOLUME': 1412469, 'OPENPOSITION': 9749764, 'SETTLEPRICE': 71318.0, 'SWAPRATE': 0.0, 'WAPRICE': 71780.0, 'CHANGE': -2.3, 'QTY': 1, 'NUMTRADES': 274693, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-20', 'SECID': 'SiM6', 'OPEN': 70890.0, 'LOW': 70550.0, 'HIGH': 72150.0, 'CLOSE': 71880.0, 'OPENPOSITIONVALUE': 692800584840.0, 'VALUE': 127111993823.0, 'VOLUME': 1773892, 'OPENPOSITION': 9628248, 'SETTLEPRICE': 71955.0, 'SWAPRATE': 0.0, 'WAPRICE': 71663.0, 'CHANGE': 1.35, 'QTY': 5, 'NUMTRADES': 329080, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-21', 'SECID': 'SiM6', 'OPEN': 71880.0, 'LOW': 71126.0, 'HIGH': 72244.0, 'CLOSE': 71706.0, 'OPENPOSITIONVALUE': 683536266432.0, 'VALUE': 98293422399.0, 'VOLUME': 1373564, 'OPENPOSITION': 9531686, 'SETTLEPRICE': 71712.0, 'SWAPRATE': 0.0, 'WAPRICE': 71536.0, 'CHANGE': -0.24, 'QTY': 1, 'NUMTRADES': 256077, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-22', 'SECID': 'SiM6', 'OPEN': 71656.0, 'LOW': 71582.0, 'HIGH': 72750.0, 'CLOSE': 72442.0, 'OPENPOSITIONVALUE': 670343045632.0, 'VALUE': 131180775718.0, 'VOLUME': 1821043, 'OPENPOSITION': 9262592, 'SETTLEPRICE': 72371.0, 'SWAPRATE': 0.0, 'WAPRICE': 72048.0, 'CHANGE': 1.03, 'QTY': 6, 'NUMTRADES': 308110, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-25', 'SECID': 'SiM6', 'OPEN': 72541.0, 'LOW': 71931.0, 'HIGH': 73198.0, 'CLOSE': 73024.0, 'OPENPOSITIONVALUE': 675771488820.0, 'VALUE': 130949490603.0, 'VOLUME': 1806996, 'OPENPOSITION': 9277860, 'SETTLEPRICE': 72837.0, 'SWAPRATE': 0.0, 'WAPRICE': 72469.0, 'CHANGE': 0.8, 'QTY': 2, 'NUMTRADES': 318389, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-26', 'SECID': 'SiM6', 'OPEN': 73024.0, 'LOW': 71671.0, 'HIGH': 73127.0, 'CLOSE': 71800.0, 'OPENPOSITIONVALUE': 665150121900.0, 'VALUE': 116864101384.0, 'VOLUME': 1615857, 'OPENPOSITION': 9267156, 'SETTLEPRICE': 71775.0, 'SWAPRATE': 0.0, 'WAPRICE': 72311.0, 'CHANGE': -1.68, 'QTY': 22, 'NUMTRADES': 253447, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-27', 'SECID': 'SiM6', 'OPEN': 71774.0, 'LOW': 71317.0, 'HIGH': 71971.0, 'CLOSE': 71805.0, 'OPENPOSITIONVALUE': 658200030552.0, 'VALUE': 113176726389.0, 'VOLUME': 1578366, 'OPENPOSITION': 9166876, 'SETTLEPRICE': 71802.0, 'SWAPRATE': 0.0, 'WAPRICE': 71700.0, 'CHANGE': 0.01, 'QTY': 8, 'NUMTRADES': 282721, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-28', 'SECID': 'SiM6', 'OPEN': 71770.0, 'LOW': 71354.0, 'HIGH': 72028.0, 'CLOSE': 71979.0, 'OPENPOSITIONVALUE': 645651244700.0, 'VALUE': 113659344611.0, 'VOLUME': 1586099, 'OPENPOSITION': 9005150, 'SETTLEPRICE': 71698.0, 'SWAPRATE': 0.0, 'WAPRICE': 71658.0, 'CHANGE': 0.24, 'QTY': 5, 'NUMTRADES': 296728, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-29', 'SECID': 'SiM6', 'OPEN': 71972.0, 'LOW': 71303.0, 'HIGH': 72167.0, 'CLOSE': 71525.0, 'OPENPOSITIONVALUE': 635918651440.0, 'VALUE': 86199266586.0, 'VOLUME': 1204188, 'OPENPOSITION': 8885020, 'SETTLEPRICE': 71572.0, 'SWAPRATE': 0.0, 'WAPRICE': 71573.0, 'CHANGE': -0.63, 'QTY': 5, 'NUMTRADES': 220797, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-01', 'SECID': 'SiM6', 'OPEN': 71518.0, 'LOW': 71425.0, 'HIGH': 72762.0, 'CLOSE': 72448.0, 'OPENPOSITIONVALUE': 641955700522.0, 'VALUE': 148847664840.0, 'VOLUME': 2064235, 'OPENPOSITION': 8863486, 'SETTLEPRICE': 72427.0, 'SWAPRATE': 0.0, 'WAPRICE': 72140.0, 'CHANGE': 1.29, 'QTY': 2, 'NUMTRADES': 324450, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-02', 'SECID': 'SiM6', 'OPEN': 72448.0, 'LOW': 72413.0, 'HIGH': 73873.0, 'CLOSE': 73835.0, 'OPENPOSITIONVALUE': 607029233038.0, 'VALUE': 137014928540.0, 'VOLUME': 1870897, 'OPENPOSITION': 8240626, 'SETTLEPRICE': 73663.0, 'SWAPRATE': 0.0, 'WAPRICE': 73291.0, 'CHANGE': 1.91, 'QTY': 1, 'NUMTRADES': 328713, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-03', 'SECID': 'SiM6', 'OPEN': 73851.0, 'LOW': 72924.0, 'HIGH': 74798.0, 'CLOSE': 74220.0, 'OPENPOSITIONVALUE': 606928291600.0, 'VALUE': 208238706374.0, 'VOLUME': 2816554, 'OPENPOSITION': 8189560, 'SETTLEPRICE': 74110.0, 'SWAPRATE': 0.0, 'WAPRICE': 73940.0, 'CHANGE': 0.52, 'QTY': 1, 'NUMTRADES': 443056, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-04', 'SECID': 'SiM6', 'OPEN': 74210.0, 'LOW': 73695.0, 'HIGH': 75097.0, 'CLOSE': 73869.0, 'OPENPOSITIONVALUE': 562980149688.0, 'VALUE': 170941878257.0, 'VOLUME': 2291595, 'OPENPOSITION': 7612572, 'SETTLEPRICE': 73954.0, 'SWAPRATE': 0.0, 'WAPRICE': 74624.0, 'CHANGE': -0.47, 'QTY': 5, 'NUMTRADES': 412531, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-05', 'SECID': 'SiM6', 'OPEN': 73882.0, 'LOW': 73785.0, 'HIGH': 74525.0, 'CLOSE': 74410.0, 'OPENPOSITIONVALUE': 539407900416.0, 'VALUE': 100816302069.0, 'VOLUME': 1359983, 'OPENPOSITION': 7250496, 'SETTLEPRICE': 74396.0, 'SWAPRATE': 0.0, 'WAPRICE': 74139.0, 'CHANGE': 0.73, 'QTY': 1, 'NUMTRADES': 235203, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-08', 'SECID': 'SiM6', 'OPEN': 74400.0, 'LOW': 73106.0, 'HIGH': 74563.0, 'CLOSE': 73151.0, 'OPENPOSITIONVALUE': 492540822116.0, 'VALUE': 118062177135.0, 'VOLUME': 1599256, 'OPENPOSITION': 6710914, 'SETTLEPRICE': 73394.0, 'SWAPRATE': 0.0, 'WAPRICE': 73758.0, 'CHANGE': -1.69, 'QTY': 3, 'NUMTRADES': 285842, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-09', 'SECID': 'SiM6', 'OPEN': 73295.0, 'LOW': 71958.0, 'HIGH': 73385.0, 'CLOSE': 72193.0, 'OPENPOSITIONVALUE': 455314279728.0, 'VALUE': 106232694146.0, 'VOLUME': 1465508, 'OPENPOSITION': 6297918, 'SETTLEPRICE': 72296.0, 'SWAPRATE': 0.0, 'WAPRICE': 72427.0, 'CHANGE': -1.31, 'QTY': 2, 'NUMTRADES': 320090, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-10', 'SECID': 'SiM6', 'OPEN': 72200.0, 'LOW': 71701.0, 'HIGH': 72800.0, 'CLOSE': 72511.0, 'OPENPOSITIONVALUE': 428405896176.0, 'VALUE': 119478854234.0, 'VOLUME': 1650200, 'OPENPOSITION': 5912472, 'SETTLEPRICE': 72458.0, 'SWAPRATE': 0.0, 'WAPRICE': 72404.0, 'CHANGE': 0.44, 'QTY': 1, 'NUMTRADES': 286828, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-11', 'SECID': 'SiM6', 'OPEN': 72500.0, 'LOW': 72072.0, 'HIGH': 72775.0, 'CLOSE': 72527.0, 'OPENPOSITIONVALUE': 401561848128.0, 'VALUE': 72138338290.0, 'VOLUME': 996401, 'OPENPOSITION': 5550114, 'SETTLEPRICE': 72352.0, 'SWAPRATE': 0.0, 'WAPRICE': 72393.0, 'CHANGE': 0.02, 'QTY': 2, 'NUMTRADES': 207191, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-15', 'SECID': 'SiM6', 'OPEN': 72834.0, 'LOW': 72188.0, 'HIGH': 73242.0, 'CLOSE': 72239.0, 'OPENPOSITIONVALUE': 320825915748.0, 'VALUE': 132768696161.0, 'VOLUME': 1829217, 'OPENPOSITION': 4437058, 'SETTLEPRICE': 72306.0, 'SWAPRATE': 0.0, 'WAPRICE': 72617.0, 'CHANGE': -0.4, 'QTY': 1, 'NUMTRADES': 329673, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-16', 'SECID': 'SiM6', 'OPEN': 72250.0, 'LOW': 71974.0, 'HIGH': 72950.0, 'CLOSE': 72817.0, 'OPENPOSITIONVALUE': 257883522012.0, 'VALUE': 148578134999.0, 'VOLUME': 2051665, 'OPENPOSITION': 3546692, 'SETTLEPRICE': 72711.0, 'SWAPRATE': 0.0, 'WAPRICE': 72436.0, 'CHANGE': 0.8, 'QTY': 1, 'NUMTRADES': 314651, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-17', 'SECID': 'SiM6', 'OPEN': 72780.0, 'LOW': 72693.0, 'HIGH': 73567.0, 'CLOSE': 73230.0, 'OPENPOSITIONVALUE': 196300669552.0, 'VALUE': 143784340577.0, 'VOLUME': 1967090, 'OPENPOSITION': 2683096, 'SETTLEPRICE': 73162.0, 'SWAPRATE': 0.0, 'WAPRICE': 73128.0, 'CHANGE': 0.57, 'QTY': 8, 'NUMTRADES': 261083, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-18', 'SECID': 'SiM6', 'OPEN': 73200.0, 'LOW': 73170.0, 'HIGH': 73775.0, 'CLOSE': 73357.0, 'OPENPOSITIONVALUE': 0.0, 'VALUE': 70089831538.0, 'VOLUME': 954751, 'OPENPOSITION': 0, 'SETTLEPRICE': 73359.0, 'SWAPRATE': 0.0, 'WAPRICE': 73463.0, 'CHANGE': 0.17, 'QTY': 11, 'NUMTRADES': 127883, 'SHORTNAME': 'Si-6.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}], 'SiU6': [{'BOARDID': 'RFUD', 'TRADEDATE': '2025-12-30', 'SECID': 'SiU6', 'OPEN': 84562.0, 'LOW': 84176.0, 'HIGH': 85450.0, 'CLOSE': 85337.0, 'OPENPOSITIONVALUE': 598936480.0, 'VALUE': 13587874.0, 'VOLUME': 160, 'OPENPOSITION': 7024, 'SETTLEPRICE': 85270.0, 'SWAPRATE': 0.0, 'WAPRICE': 84924.0, 'CHANGE': 0.84, 'QTY': 1, 'NUMTRADES': 142, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-05', 'SECID': 'SiU6', 'OPEN': 85329.0, 'LOW': 85149.0, 'HIGH': 86581.0, 'CLOSE': 86348.0, 'OPENPOSITIONVALUE': 601697812.0, 'VALUE': 16963443.0, 'VOLUME': 197, 'OPENPOSITION': 6962, 'SETTLEPRICE': 86426.0, 'SWAPRATE': 0.0, 'WAPRICE': 86095.0, 'CHANGE': 1.19, 'QTY': 1, 'NUMTRADES': 158, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-06', 'SECID': 'SiU6', 'OPEN': 86548.0, 'LOW': 85790.0, 'HIGH': 86760.0, 'CLOSE': 85888.0, 'OPENPOSITIONVALUE': 599841792.0, 'VALUE': 11985760.0, 'VOLUME': 139, 'OPENPOSITION': 6984, 'SETTLEPRICE': 85888.0, 'SWAPRATE': 0.0, 'WAPRICE': 86220.0, 'CHANGE': -0.53, 'QTY': 1, 'NUMTRADES': 126, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-08', 'SECID': 'SiU6', 'OPEN': 85824.0, 'LOW': 85793.0, 'HIGH': 87098.0, 'CLOSE': 85898.0, 'OPENPOSITIONVALUE': 631000848.0, 'VALUE': 61025672.0, 'VOLUME': 708, 'OPENPOSITION': 7342, 'SETTLEPRICE': 85944.0, 'SWAPRATE': 0.0, 'WAPRICE': 86199.0, 'CHANGE': 0.01, 'QTY': 1, 'NUMTRADES': 391, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-09', 'SECID': 'SiU6', 'OPEN': 86039.0, 'LOW': 85202.0, 'HIGH': 86139.0, 'CLOSE': 85240.0, 'OPENPOSITIONVALUE': 633844640.0, 'VALUE': 14534696.0, 'VOLUME': 170, 'OPENPOSITION': 7436, 'SETTLEPRICE': 85240.0, 'SWAPRATE': 0.0, 'WAPRICE': 85498.0, 'CHANGE': -0.77, 'QTY': 1, 'NUMTRADES': 133, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-12', 'SECID': 'SiU6', 'OPEN': 85252.0, 'LOW': 85006.0, 'HIGH': 85760.0, 'CLOSE': 85465.0, 'OPENPOSITIONVALUE': 641158430.0, 'VALUE': 22621396.0, 'VOLUME': 265, 'OPENPOSITION': 7502, 'SETTLEPRICE': 85465.0, 'SWAPRATE': 0.0, 'WAPRICE': 85367.0, 'CHANGE': 0.26, 'QTY': 1, 'NUMTRADES': 214, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-13', 'SECID': 'SiU6', 'OPEN': 85402.0, 'LOW': 85185.0, 'HIGH': 85620.0, 'CLOSE': 85361.0, 'OPENPOSITIONVALUE': 642256164.0, 'VALUE': 6234040.0, 'VOLUME': 73, 'OPENPOSITION': 7524, 'SETTLEPRICE': 85361.0, 'SWAPRATE': 0.0, 'WAPRICE': 85398.0, 'CHANGE': -0.12, 'QTY': 1, 'NUMTRADES': 65, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-14', 'SECID': 'SiU6', 'OPEN': 85510.0, 'LOW': 84746.0, 'HIGH': 85510.0, 'CLOSE': 84859.0, 'OPENPOSITIONVALUE': 641534040.0, 'VALUE': 5867956.0, 'VOLUME': 69, 'OPENPOSITION': 7560, 'SETTLEPRICE': 84859.0, 'SWAPRATE': 0.0, 'WAPRICE': 85043.0, 'CHANGE': -0.59, 'QTY': 3, 'NUMTRADES': 58, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-15', 'SECID': 'SiU6', 'OPEN': 84848.0, 'LOW': 84291.0, 'HIGH': 85337.0, 'CLOSE': 84445.0, 'OPENPOSITIONVALUE': 645328690.0, 'VALUE': 22814247.0, 'VOLUME': 269, 'OPENPOSITION': 7642, 'SETTLEPRICE': 84445.0, 'SWAPRATE': 0.0, 'WAPRICE': 84811.0, 'CHANGE': -0.49, 'QTY': 1, 'NUMTRADES': 210, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-16', 'SECID': 'SiU6', 'OPEN': 84447.0, 'LOW': 84090.0, 'HIGH': 84620.0, 'CLOSE': 84613.0, 'OPENPOSITIONVALUE': 654396942.0, 'VALUE': 20225739.0, 'VOLUME': 240, 'OPENPOSITION': 7734, 'SETTLEPRICE': 84613.0, 'SWAPRATE': 0.0, 'WAPRICE': 84273.0, 'CHANGE': 0.2, 'QTY': 1, 'NUMTRADES': 161, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-19', 'SECID': 'SiU6', 'OPEN': 84531.0, 'LOW': 83991.0, 'HIGH': 84622.0, 'CLOSE': 84138.0, 'OPENPOSITIONVALUE': 664690200.0, 'VALUE': 27112004.0, 'VOLUME': 322, 'OPENPOSITION': 7900, 'SETTLEPRICE': 84138.0, 'SWAPRATE': 0.0, 'WAPRICE': 84197.0, 'CHANGE': -0.56, 'QTY': 2, 'NUMTRADES': 210, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-20', 'SECID': 'SiU6', 'OPEN': 84114.0, 'LOW': 83854.0, 'HIGH': 84662.0, 'CLOSE': 84244.0, 'OPENPOSITIONVALUE': 761078456.0, 'VALUE': 62848294.0, 'VOLUME': 745, 'OPENPOSITION': 9028, 'SETTLEPRICE': 84302.0, 'SWAPRATE': 0.0, 'WAPRICE': 84361.0, 'CHANGE': 0.13, 'QTY': 2, 'NUMTRADES': 276, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-21', 'SECID': 'SiU6', 'OPEN': 84275.0, 'LOW': 83100.0, 'HIGH': 84529.0, 'CLOSE': 83115.0, 'OPENPOSITIONVALUE': 944626552.0, 'VALUE': 186911668.0, 'VOLUME': 2225, 'OPENPOSITION': 11372, 'SETTLEPRICE': 83066.0, 'SWAPRATE': 0.0, 'WAPRICE': 83856.0, 'CHANGE': -1.34, 'QTY': 1, 'NUMTRADES': 555, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-22', 'SECID': 'SiU6', 'OPEN': 83007.0, 'LOW': 81984.0, 'HIGH': 83260.0, 'CLOSE': 82187.0, 'OPENPOSITIONVALUE': 1371774096.0, 'VALUE': 625470462.0, 'VOLUME': 7584, 'OPENPOSITION': 16698, 'SETTLEPRICE': 82152.0, 'SWAPRATE': 0.0, 'WAPRICE': 82304.0, 'CHANGE': -1.12, 'QTY': 1, 'NUMTRADES': 1248, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-23', 'SECID': 'SiU6', 'OPEN': 82164.0, 'LOW': 81828.0, 'HIGH': 83000.0, 'CLOSE': 82225.0, 'OPENPOSITIONVALUE': 1393825500.0, 'VALUE': 47096884.0, 'VOLUME': 573, 'OPENPOSITION': 16972, 'SETTLEPRICE': 82125.0, 'SWAPRATE': 0.0, 'WAPRICE': 82208.0, 'CHANGE': 0.05, 'QTY': 1, 'NUMTRADES': 354, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-26', 'SECID': 'SiU6', 'OPEN': 82083.0, 'LOW': 81554.0, 'HIGH': 82702.0, 'CLOSE': 82700.0, 'OPENPOSITIONVALUE': 1363246722.0, 'VALUE': 143375969.0, 'VOLUME': 1747, 'OPENPOSITION': 16502, 'SETTLEPRICE': 82611.0, 'SWAPRATE': 0.0, 'WAPRICE': 82036.0, 'CHANGE': 0.58, 'QTY': 1, 'NUMTRADES': 777, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-27', 'SECID': 'SiU6', 'OPEN': 82648.0, 'LOW': 81840.0, 'HIGH': 82905.0, 'CLOSE': 81927.0, 'OPENPOSITIONVALUE': 1281898800.0, 'VALUE': 267893980.0, 'VOLUME': 3257, 'OPENPOSITION': 15652, 'SETTLEPRICE': 81900.0, 'SWAPRATE': 0.0, 'WAPRICE': 82246.0, 'CHANGE': -0.94, 'QTY': 1, 'NUMTRADES': 705, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-28', 'SECID': 'SiU6', 'OPEN': 81913.0, 'LOW': 81186.0, 'HIGH': 82158.0, 'CLOSE': 82158.0, 'OPENPOSITIONVALUE': 1524337020.0, 'VALUE': 263490976.0, 'VOLUME': 3225, 'OPENPOSITION': 18570, 'SETTLEPRICE': 82086.0, 'SWAPRATE': 0.0, 'WAPRICE': 81703.0, 'CHANGE': 0.28, 'QTY': 1, 'NUMTRADES': 1446, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-29', 'SECID': 'SiU6', 'OPEN': 81962.0, 'LOW': 80740.0, 'HIGH': 82067.0, 'CLOSE': 80833.0, 'OPENPOSITIONVALUE': 1357832734.0, 'VALUE': 167826543.0, 'VOLUME': 2062, 'OPENPOSITION': 16798, 'SETTLEPRICE': 80833.0, 'SWAPRATE': 0.0, 'WAPRICE': 81319.0, 'CHANGE': -1.61, 'QTY': 1, 'NUMTRADES': 630, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-01-30', 'SECID': 'SiU6', 'OPEN': 80914.0, 'LOW': 80830.0, 'HIGH': 81700.0, 'CLOSE': 81500.0, 'OPENPOSITIONVALUE': 1866558804.0, 'VALUE': 417246245.0, 'VOLUME': 5145, 'OPENPOSITION': 22902, 'SETTLEPRICE': 81502.0, 'SWAPRATE': 0.0, 'WAPRICE': 81403.0, 'CHANGE': 0.83, 'QTY': 2, 'NUMTRADES': 1129, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-02', 'SECID': 'SiU6', 'OPEN': 81608.0, 'LOW': 81446.0, 'HIGH': 82837.0, 'CLOSE': 81903.0, 'OPENPOSITIONVALUE': 1967219682.0, 'VALUE': 94108952.0, 'VOLUME': 1143, 'OPENPOSITION': 24006, 'SETTLEPRICE': 81947.0, 'SWAPRATE': 0.0, 'WAPRICE': 82342.0, 'CHANGE': 0.49, 'QTY': 1, 'NUMTRADES': 521, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-03', 'SECID': 'SiU6', 'OPEN': 81990.0, 'LOW': 81900.0, 'HIGH': 82250.0, 'CLOSE': 82183.0, 'OPENPOSITIONVALUE': 2149742914.0, 'VALUE': 128042380.0, 'VOLUME': 1560, 'OPENPOSITION': 26158, 'SETTLEPRICE': 82183.0, 'SWAPRATE': 0.0, 'WAPRICE': 82099.0, 'CHANGE': 0.34, 'QTY': 1, 'NUMTRADES': 517, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-04', 'SECID': 'SiU6', 'OPEN': 82223.0, 'LOW': 81380.0, 'HIGH': 82500.0, 'CLOSE': 81547.0, 'OPENPOSITIONVALUE': 2145612040.0, 'VALUE': 53780657.0, 'VOLUME': 657, 'OPENPOSITION': 26312, 'SETTLEPRICE': 81545.0, 'SWAPRATE': 0.0, 'WAPRICE': 81839.0, 'CHANGE': -0.77, 'QTY': 1, 'NUMTRADES': 416, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-05', 'SECID': 'SiU6', 'OPEN': 81597.0, 'LOW': 81034.0, 'HIGH': 81876.0, 'CLOSE': 81700.0, 'OPENPOSITIONVALUE': 2191030600.0, 'VALUE': 51169002.0, 'VOLUME': 627, 'OPENPOSITION': 26818, 'SETTLEPRICE': 81700.0, 'SWAPRATE': 0.0, 'WAPRICE': 81607.0, 'CHANGE': 0.19, 'QTY': 1, 'NUMTRADES': 371, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-06', 'SECID': 'SiU6', 'OPEN': 81720.0, 'LOW': 81240.0, 'HIGH': 82249.0, 'CLOSE': 81715.0, 'OPENPOSITIONVALUE': 2462563240.0, 'VALUE': 164465414.0, 'VOLUME': 2012, 'OPENPOSITION': 30136, 'SETTLEPRICE': 81715.0, 'SWAPRATE': 0.0, 'WAPRICE': 81743.0, 'CHANGE': 0.02, 'QTY': 2, 'NUMTRADES': 515, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-09', 'SECID': 'SiU6', 'OPEN': 81765.0, 'LOW': 81641.0, 'HIGH': 82218.0, 'CLOSE': 82010.0, 'OPENPOSITIONVALUE': 2679440400.0, 'VALUE': 286152086.0, 'VOLUME': 3490, 'OPENPOSITION': 32716, 'SETTLEPRICE': 81900.0, 'SWAPRATE': 0.0, 'WAPRICE': 81993.0, 'CHANGE': 0.36, 'QTY': 1, 'NUMTRADES': 486, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-10', 'SECID': 'SiU6', 'OPEN': 81922.0, 'LOW': 81650.0, 'HIGH': 82064.0, 'CLOSE': 81840.0, 'OPENPOSITIONVALUE': 2696300640.0, 'VALUE': 40366196.0, 'VOLUME': 493, 'OPENPOSITION': 32946, 'SETTLEPRICE': 81840.0, 'SWAPRATE': 0.0, 'WAPRICE': 81879.0, 'CHANGE': -0.21, 'QTY': 1, 'NUMTRADES': 188, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-11', 'SECID': 'SiU6', 'OPEN': 81952.0, 'LOW': 81400.0, 'HIGH': 82271.0, 'CLOSE': 81618.0, 'OPENPOSITIONVALUE': 2688660156.0, 'VALUE': 28661543.0, 'VOLUME': 351, 'OPENPOSITION': 32942, 'SETTLEPRICE': 81618.0, 'SWAPRATE': 0.0, 'WAPRICE': 81660.0, 'CHANGE': -0.27, 'QTY': 1, 'NUMTRADES': 228, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-12', 'SECID': 'SiU6', 'OPEN': 81698.0, 'LOW': 81360.0, 'HIGH': 81750.0, 'CLOSE': 81569.0, 'OPENPOSITIONVALUE': 2692184594.0, 'VALUE': 14120608.0, 'VOLUME': 173, 'OPENPOSITION': 33022, 'SETTLEPRICE': 81527.0, 'SWAPRATE': 0.0, 'WAPRICE': 81624.0, 'CHANGE': -0.06, 'QTY': 1, 'NUMTRADES': 129, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-13', 'SECID': 'SiU6', 'OPEN': 81465.0, 'LOW': 80601.0, 'HIGH': 81597.0, 'CLOSE': 80636.0, 'OPENPOSITIONVALUE': 2666793792.0, 'VALUE': 56439604.0, 'VOLUME': 697, 'OPENPOSITION': 33072, 'SETTLEPRICE': 80636.0, 'SWAPRATE': 0.0, 'WAPRICE': 80971.0, 'CHANGE': -1.14, 'QTY': 1, 'NUMTRADES': 434, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-16', 'SECID': 'SiU6', 'OPEN': 80602.0, 'LOW': 80350.0, 'HIGH': 80889.0, 'CLOSE': 80889.0, 'OPENPOSITIONVALUE': 2729868750.0, 'VALUE': 48659844.0, 'VOLUME': 604, 'OPENPOSITION': 33750, 'SETTLEPRICE': 80885.0, 'SWAPRATE': 0.0, 'WAPRICE': 80563.0, 'CHANGE': 0.31, 'QTY': 1, 'NUMTRADES': 296, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-17', 'SECID': 'SiU6', 'OPEN': 80839.0, 'LOW': 80119.0, 'HIGH': 81199.0, 'CLOSE': 80164.0, 'OPENPOSITIONVALUE': 2769364884.0, 'VALUE': 61408672.0, 'VOLUME': 764, 'OPENPOSITION': 34554, 'SETTLEPRICE': 80146.0, 'SWAPRATE': 0.0, 'WAPRICE': 80362.0, 'CHANGE': -0.9, 'QTY': 1, 'NUMTRADES': 310, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-18', 'SECID': 'SiU6', 'OPEN': 80129.0, 'LOW': 79542.0, 'HIGH': 80785.0, 'CLOSE': 80600.0, 'OPENPOSITIONVALUE': 2817937200.0, 'VALUE': 94371129.0, 'VOLUME': 1180, 'OPENPOSITION': 34962, 'SETTLEPRICE': 80600.0, 'SWAPRATE': 0.0, 'WAPRICE': 79975.0, 'CHANGE': 0.54, 'QTY': 1, 'NUMTRADES': 546, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-19', 'SECID': 'SiU6', 'OPEN': 80600.0, 'LOW': 79746.0, 'HIGH': 80715.0, 'CLOSE': 79838.0, 'OPENPOSITIONVALUE': 2790497776.0, 'VALUE': 26386710.0, 'VOLUME': 329, 'OPENPOSITION': 34952, 'SETTLEPRICE': 79838.0, 'SWAPRATE': 0.0, 'WAPRICE': 80183.0, 'CHANGE': -0.95, 'QTY': 1, 'NUMTRADES': 268, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-20', 'SECID': 'SiU6', 'OPEN': 79894.0, 'LOW': 79500.0, 'HIGH': 80086.0, 'CLOSE': 79960.0, 'OPENPOSITIONVALUE': 3255226128.0, 'VALUE': 317455340.0, 'VOLUME': 3971, 'OPENPOSITION': 40728, 'SETTLEPRICE': 79926.0, 'SWAPRATE': 0.0, 'WAPRICE': 79945.0, 'CHANGE': 0.15, 'QTY': 1, 'NUMTRADES': 954, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-24', 'SECID': 'SiU6', 'OPEN': 79885.0, 'LOW': 79020.0, 'HIGH': 79999.0, 'CLOSE': 79187.0, 'OPENPOSITIONVALUE': 3308659200.0, 'VALUE': 115894680.0, 'VOLUME': 1458, 'OPENPOSITION': 41776, 'SETTLEPRICE': 79200.0, 'SWAPRATE': 0.0, 'WAPRICE': 79532.0, 'CHANGE': -0.97, 'QTY': 1, 'NUMTRADES': 697, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-25', 'SECID': 'SiU6', 'OPEN': 79258.0, 'LOW': 79010.0, 'HIGH': 81180.0, 'CLOSE': 80700.0, 'OPENPOSITIONVALUE': 3503832600.0, 'VALUE': 158447298.0, 'VOLUME': 1977, 'OPENPOSITION': 43418, 'SETTLEPRICE': 80700.0, 'SWAPRATE': 0.0, 'WAPRICE': 80243.0, 'CHANGE': 1.91, 'QTY': 2, 'NUMTRADES': 924, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-26', 'SECID': 'SiU6', 'OPEN': 80751.0, 'LOW': 80000.0, 'HIGH': 80919.0, 'CLOSE': 80100.0, 'OPENPOSITIONVALUE': 3740196432.0, 'VALUE': 228607280.0, 'VOLUME': 2836, 'OPENPOSITION': 46704, 'SETTLEPRICE': 80083.0, 'SWAPRATE': 0.0, 'WAPRICE': 80594.0, 'CHANGE': -0.74, 'QTY': 1, 'NUMTRADES': 992, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-02-27', 'SECID': 'SiU6', 'OPEN': 80100.0, 'LOW': 80005.0, 'HIGH': 80633.0, 'CLOSE': 80005.0, 'OPENPOSITIONVALUE': 3831276900.0, 'VALUE': 188633630.0, 'VOLUME': 2345, 'OPENPOSITION': 47876, 'SETTLEPRICE': 80025.0, 'SWAPRATE': 0.0, 'WAPRICE': 80441.0, 'CHANGE': -0.12, 'QTY': 1, 'NUMTRADES': 872, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-02', 'SECID': 'SiU6', 'OPEN': 80001.0, 'LOW': 79089.0, 'HIGH': 80902.0, 'CLOSE': 80861.0, 'OPENPOSITIONVALUE': 4296328506.0, 'VALUE': 452449123.0, 'VOLUME': 5624, 'OPENPOSITION': 53106, 'SETTLEPRICE': 80901.0, 'SWAPRATE': 0.0, 'WAPRICE': 80453.0, 'CHANGE': 1.07, 'QTY': 1, 'NUMTRADES': 1400, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-03', 'SECID': 'SiU6', 'OPEN': 80901.0, 'LOW': 80295.0, 'HIGH': 80999.0, 'CLOSE': 80513.0, 'OPENPOSITIONVALUE': 8213453182.0, 'VALUE': 2206587758.0, 'VOLUME': 27401, 'OPENPOSITION': 102014, 'SETTLEPRICE': 80513.0, 'SWAPRATE': 0.0, 'WAPRICE': 80530.0, 'CHANGE': -0.43, 'QTY': 1, 'NUMTRADES': 5546, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-04', 'SECID': 'SiU6', 'OPEN': 80510.0, 'LOW': 80305.0, 'HIGH': 81697.0, 'CLOSE': 81240.0, 'OPENPOSITIONVALUE': 13318771144.0, 'VALUE': 2742904844.0, 'VOLUME': 33852, 'OPENPOSITION': 163996, 'SETTLEPRICE': 81214.0, 'SWAPRATE': 0.0, 'WAPRICE': 81102.0, 'CHANGE': 0.9, 'QTY': 5, 'NUMTRADES': 2293, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-05', 'SECID': 'SiU6', 'OPEN': 81262.0, 'LOW': 81116.0, 'HIGH': 83037.0, 'CLOSE': 82775.0, 'OPENPOSITIONVALUE': 18560441032.0, 'VALUE': 2720831717.0, 'VOLUME': 33163, 'OPENPOSITION': 224252, 'SETTLEPRICE': 82766.0, 'SWAPRATE': 0.0, 'WAPRICE': 82086.0, 'CHANGE': 1.89, 'QTY': 1, 'NUMTRADES': 4498, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-06', 'SECID': 'SiU6', 'OPEN': 82750.0, 'LOW': 81879.0, 'HIGH': 82999.0, 'CLOSE': 81965.0, 'OPENPOSITIONVALUE': 19240907348.0, 'VALUE': 683086401.0, 'VOLUME': 8271, 'OPENPOSITION': 234754, 'SETTLEPRICE': 81962.0, 'SWAPRATE': 0.0, 'WAPRICE': 82574.0, 'CHANGE': -0.98, 'QTY': 1, 'NUMTRADES': 2126, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-09', 'SECID': 'SiU6', 'OPEN': 81905.0, 'LOW': 81025.0, 'HIGH': 82206.0, 'CLOSE': 81300.0, 'OPENPOSITIONVALUE': 19099971600.0, 'VALUE': 117795668.0, 'VOLUME': 1445, 'OPENPOSITION': 234932, 'SETTLEPRICE': 81300.0, 'SWAPRATE': 0.0, 'WAPRICE': 81518.0, 'CHANGE': -0.81, 'QTY': 1, 'NUMTRADES': 1014, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-10', 'SECID': 'SiU6', 'OPEN': 81300.0, 'LOW': 81249.0, 'HIGH': 82075.0, 'CLOSE': 81717.0, 'OPENPOSITIONVALUE': 19541910196.0, 'VALUE': 296909216.0, 'VOLUME': 3636, 'OPENPOSITION': 239276, 'SETTLEPRICE': 81671.0, 'SWAPRATE': 0.0, 'WAPRICE': 81666.0, 'CHANGE': 0.51, 'QTY': 1, 'NUMTRADES': 1173, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-11', 'SECID': 'SiU6', 'OPEN': 81730.0, 'LOW': 81100.0, 'HIGH': 81999.0, 'CLOSE': 81519.0, 'OPENPOSITIONVALUE': 19843270020.0, 'VALUE': 509621571.0, 'VOLUME': 6262, 'OPENPOSITION': 243410, 'SETTLEPRICE': 81522.0, 'SWAPRATE': 0.0, 'WAPRICE': 81341.0, 'CHANGE': -0.24, 'QTY': 1, 'NUMTRADES': 1752, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-12', 'SECID': 'SiU6', 'OPEN': 81480.0, 'LOW': 80998.0, 'HIGH': 82133.0, 'CLOSE': 81916.0, 'OPENPOSITIONVALUE': 20134297292.0, 'VALUE': 191584219.0, 'VOLUME': 2349, 'OPENPOSITION': 245738, 'SETTLEPRICE': 81934.0, 'SWAPRATE': 0.0, 'WAPRICE': 81560.0, 'CHANGE': 0.49, 'QTY': 1, 'NUMTRADES': 836, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-13', 'SECID': 'SiU6', 'OPEN': 81950.0, 'LOW': 81886.0, 'HIGH': 82736.0, 'CLOSE': 82496.0, 'OPENPOSITIONVALUE': 20432115600.0, 'VALUE': 189682373.0, 'VOLUME': 2303, 'OPENPOSITION': 247512, 'SETTLEPRICE': 82550.0, 'SWAPRATE': 0.0, 'WAPRICE': 82365.0, 'CHANGE': 0.71, 'QTY': 2, 'NUMTRADES': 888, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-16', 'SECID': 'SiU6', 'OPEN': 82632.0, 'LOW': 82300.0, 'HIGH': 83378.0, 'CLOSE': 83372.0, 'OPENPOSITIONVALUE': 20785643232.0, 'VALUE': 375274456.0, 'VOLUME': 4527, 'OPENPOSITION': 249324, 'SETTLEPRICE': 83368.0, 'SWAPRATE': 0.0, 'WAPRICE': 82898.0, 'CHANGE': 1.06, 'QTY': 2, 'NUMTRADES': 1731, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-17', 'SECID': 'SiU6', 'OPEN': 83399.0, 'LOW': 83175.0, 'HIGH': 84623.0, 'CLOSE': 84617.0, 'OPENPOSITIONVALUE': 22923796192.0, 'VALUE': 1365372205.0, 'VOLUME': 16267, 'OPENPOSITION': 271172, 'SETTLEPRICE': 84536.0, 'SWAPRATE': 0.0, 'WAPRICE': 84021.0, 'CHANGE': 1.49, 'QTY': 5, 'NUMTRADES': 4292, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-18', 'SECID': 'SiU6', 'OPEN': 84614.0, 'LOW': 83739.0, 'HIGH': 84990.0, 'CLOSE': 84591.0, 'OPENPOSITIONVALUE': 25804750806.0, 'VALUE': 2342149585.0, 'VOLUME': 27707, 'OPENPOSITION': 305082, 'SETTLEPRICE': 84583.0, 'SWAPRATE': 0.0, 'WAPRICE': 84585.0, 'CHANGE': -0.03, 'QTY': 1, 'NUMTRADES': 7740, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-19', 'SECID': 'SiU6', 'OPEN': 84591.0, 'LOW': 84076.0, 'HIGH': 85766.0, 'CLOSE': 84594.0, 'OPENPOSITIONVALUE': 28414100738.0, 'VALUE': 3035833094.0, 'VOLUME': 35762, 'OPENPOSITION': 336178, 'SETTLEPRICE': 84521.0, 'SWAPRATE': 0.0, 'WAPRICE': 84904.0, 'CHANGE': 0.0, 'QTY': 2, 'NUMTRADES': 9674, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-20', 'SECID': 'SiU6', 'OPEN': 84506.0, 'LOW': 83717.0, 'HIGH': 85225.0, 'CLOSE': 84317.0, 'OPENPOSITIONVALUE': 29318229696.0, 'VALUE': 2352007079.0, 'VOLUME': 27855, 'OPENPOSITION': 347636, 'SETTLEPRICE': 84336.0, 'SWAPRATE': 0.0, 'WAPRICE': 84413.0, 'CHANGE': -0.33, 'QTY': 3, 'NUMTRADES': 10984, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-23', 'SECID': 'SiU6', 'OPEN': 84362.0, 'LOW': 82823.0, 'HIGH': 84896.0, 'CLOSE': 82937.0, 'OPENPOSITIONVALUE': 31455759090.0, 'VALUE': 2209834704.0, 'VOLUME': 26378, 'OPENPOSITION': 374942, 'SETTLEPRICE': 83895.0, 'SWAPRATE': 0.0, 'WAPRICE': 83756.0, 'CHANGE': -1.64, 'QTY': 1, 'NUMTRADES': 8584, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-24', 'SECID': 'SiU6', 'OPEN': 82801.0, 'LOW': 81500.0, 'HIGH': 83143.0, 'CLOSE': 81917.0, 'OPENPOSITIONVALUE': 32302761792.0, 'VALUE': 1611027173.0, 'VOLUME': 19560, 'OPENPOSITION': 395402, 'SETTLEPRICE': 81696.0, 'SWAPRATE': 0.0, 'WAPRICE': 82322.0, 'CHANGE': -1.23, 'QTY': 1, 'NUMTRADES': 6639, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-25', 'SECID': 'SiU6', 'OPEN': 81920.0, 'LOW': 81670.0, 'HIGH': 83038.0, 'CLOSE': 82770.0, 'OPENPOSITIONVALUE': 34025230686.0, 'VALUE': 1562474384.0, 'VOLUME': 18970, 'OPENPOSITION': 410898, 'SETTLEPRICE': 82807.0, 'SWAPRATE': 0.0, 'WAPRICE': 82389.0, 'CHANGE': 1.04, 'QTY': 1, 'NUMTRADES': 7595, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-26', 'SECID': 'SiU6', 'OPEN': 82980.0, 'LOW': 81726.0, 'HIGH': 83734.0, 'CLOSE': 82152.0, 'OPENPOSITIONVALUE': 34126810970.0, 'VALUE': 1731784471.0, 'VOLUME': 20910, 'OPENPOSITION': 416206, 'SETTLEPRICE': 81995.0, 'SWAPRATE': 0.0, 'WAPRICE': 82824.0, 'CHANGE': -0.75, 'QTY': 3, 'NUMTRADES': 9297, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-27', 'SECID': 'SiU6', 'OPEN': 82000.0, 'LOW': 81692.0, 'HIGH': 83000.0, 'CLOSE': 82865.0, 'OPENPOSITIONVALUE': 35450507544.0, 'VALUE': 1454760765.0, 'VOLUME': 17670, 'OPENPOSITION': 427506, 'SETTLEPRICE': 82924.0, 'SWAPRATE': 0.0, 'WAPRICE': 82364.0, 'CHANGE': 0.87, 'QTY': 1, 'NUMTRADES': 8233, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-30', 'SECID': 'SiU6', 'OPEN': 82616.0, 'LOW': 82400.0, 'HIGH': 83468.0, 'CLOSE': 82909.0, 'OPENPOSITIONVALUE': 35836467548.0, 'VALUE': 1045255697.0, 'VOLUME': 12620, 'OPENPOSITION': 430556, 'SETTLEPRICE': 83233.0, 'SWAPRATE': 0.0, 'WAPRICE': 82809.0, 'CHANGE': 0.05, 'QTY': 1, 'NUMTRADES': 6855, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-03-31', 'SECID': 'SiU6', 'OPEN': 82950.0, 'LOW': 82702.0, 'HIGH': 83800.0, 'CLOSE': 83557.0, 'OPENPOSITIONVALUE': 37081932300.0, 'VALUE': 1797883299.0, 'VOLUME': 21541, 'OPENPOSITION': 444228, 'SETTLEPRICE': 83475.0, 'SWAPRATE': 0.0, 'WAPRICE': 83470.0, 'CHANGE': 0.78, 'QTY': 1, 'NUMTRADES': 8579, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-01', 'SECID': 'SiU6', 'OPEN': 83526.0, 'LOW': 81880.0, 'HIGH': 83526.0, 'CLOSE': 82214.0, 'OPENPOSITIONVALUE': 37466812044.0, 'VALUE': 1722711241.0, 'VOLUME': 20842, 'OPENPOSITION': 456228, 'SETTLEPRICE': 82123.0, 'SWAPRATE': 0.0, 'WAPRICE': 82320.0, 'CHANGE': -1.61, 'QTY': 4, 'NUMTRADES': 6954, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-02', 'SECID': 'SiU6', 'OPEN': 82214.0, 'LOW': 81733.0, 'HIGH': 82450.0, 'CLOSE': 81912.0, 'OPENPOSITIONVALUE': 38453118982.0, 'VALUE': 1457602613.0, 'VOLUME': 17766, 'OPENPOSITION': 469622, 'SETTLEPRICE': 81881.0, 'SWAPRATE': 0.0, 'WAPRICE': 82064.0, 'CHANGE': -0.37, 'QTY': 1, 'NUMTRADES': 6507, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-03', 'SECID': 'SiU6', 'OPEN': 81801.0, 'LOW': 80850.0, 'HIGH': 81901.0, 'CLOSE': 80957.0, 'OPENPOSITIONVALUE': 39374091134.0, 'VALUE': 1676953147.0, 'VOLUME': 20568, 'OPENPOSITION': 486286, 'SETTLEPRICE': 80969.0, 'SWAPRATE': 0.0, 'WAPRICE': 81379.0, 'CHANGE': -1.17, 'QTY': 1, 'NUMTRADES': 5799, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-06', 'SECID': 'SiU6', 'OPEN': 80801.0, 'LOW': 80291.0, 'HIGH': 81065.0, 'CLOSE': 80808.0, 'OPENPOSITIONVALUE': 39583856520.0, 'VALUE': 805639864.0, 'VOLUME': 9980, 'OPENPOSITION': 490324, 'SETTLEPRICE': 80730.0, 'SWAPRATE': 0.0, 'WAPRICE': 80702.0, 'CHANGE': -0.18, 'QTY': 1, 'NUMTRADES': 4766, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-07', 'SECID': 'SiU6', 'OPEN': 80808.0, 'LOW': 80101.0, 'HIGH': 81064.0, 'CLOSE': 80545.0, 'OPENPOSITIONVALUE': 41295343320.0, 'VALUE': 2468918345.0, 'VOLUME': 30647, 'OPENPOSITION': 512820, 'SETTLEPRICE': 80526.0, 'SWAPRATE': 0.0, 'WAPRICE': 80537.0, 'CHANGE': -0.33, 'QTY': 2, 'NUMTRADES': 8565, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-08', 'SECID': 'SiU6', 'OPEN': 80777.0, 'LOW': 80236.0, 'HIGH': 80940.0, 'CLOSE': 80591.0, 'OPENPOSITIONVALUE': 42649384320.0, 'VALUE': 2303966014.0, 'VOLUME': 28607, 'OPENPOSITION': 528624, 'SETTLEPRICE': 80680.0, 'SWAPRATE': 0.0, 'WAPRICE': 80520.0, 'CHANGE': 0.06, 'QTY': 2, 'NUMTRADES': 10692, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-09', 'SECID': 'SiU6', 'OPEN': 80515.0, 'LOW': 79599.0, 'HIGH': 80700.0, 'CLOSE': 79615.0, 'OPENPOSITIONVALUE': 45828088774.0, 'VALUE': 3518506972.0, 'VOLUME': 43783, 'OPENPOSITION': 574366, 'SETTLEPRICE': 79789.0, 'SWAPRATE': 0.0, 'WAPRICE': 80251.0, 'CHANGE': -1.21, 'QTY': 1, 'NUMTRADES': 12424, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-10', 'SECID': 'SiU6', 'OPEN': 79566.0, 'LOW': 78849.0, 'HIGH': 79793.0, 'CLOSE': 79080.0, 'OPENPOSITIONVALUE': 50863348182.0, 'VALUE': 4123930644.0, 'VOLUME': 51958, 'OPENPOSITION': 643278, 'SETTLEPRICE': 79069.0, 'SWAPRATE': 0.0, 'WAPRICE': 79177.0, 'CHANGE': -0.67, 'QTY': 1, 'NUMTRADES': 10972, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-13', 'SECID': 'SiU6', 'OPEN': 78800.0, 'LOW': 78150.0, 'HIGH': 78992.0, 'CLOSE': 78392.0, 'OPENPOSITIONVALUE': 51647511264.0, 'VALUE': 1754429735.0, 'VOLUME': 22332, 'OPENPOSITION': 656976, 'SETTLEPRICE': 78614.0, 'SWAPRATE': 0.0, 'WAPRICE': 78425.0, 'CHANGE': -0.87, 'QTY': 9, 'NUMTRADES': 8239, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-14', 'SECID': 'SiU6', 'OPEN': 78522.0, 'LOW': 77123.0, 'HIGH': 78522.0, 'CLOSE': 77175.0, 'OPENPOSITIONVALUE': 53029290960.0, 'VALUE': 3061940795.0, 'VOLUME': 39321, 'OPENPOSITION': 686410, 'SETTLEPRICE': 77256.0, 'SWAPRATE': 0.0, 'WAPRICE': 77751.0, 'CHANGE': -1.55, 'QTY': 5, 'NUMTRADES': 10990, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-15', 'SECID': 'SiU6', 'OPEN': 77256.0, 'LOW': 76920.0, 'HIGH': 78380.0, 'CLOSE': 78018.0, 'OPENPOSITIONVALUE': 58404714960.0, 'VALUE': 4653139625.0, 'VOLUME': 60045, 'OPENPOSITION': 748740, 'SETTLEPRICE': 78004.0, 'SWAPRATE': 0.0, 'WAPRICE': 77643.0, 'CHANGE': 1.09, 'QTY': 1, 'NUMTRADES': 14277, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-16', 'SECID': 'SiU6', 'OPEN': 78100.0, 'LOW': 77591.0, 'HIGH': 79170.0, 'CLOSE': 78440.0, 'OPENPOSITIONVALUE': 59888308672.0, 'VALUE': 2009468389.0, 'VOLUME': 25563, 'OPENPOSITION': 764818, 'SETTLEPRICE': 78304.0, 'SWAPRATE': 0.0, 'WAPRICE': 78701.0, 'CHANGE': 0.54, 'QTY': 1, 'NUMTRADES': 8927, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-17', 'SECID': 'SiU6', 'OPEN': 78440.0, 'LOW': 77784.0, 'HIGH': 78824.0, 'CLOSE': 78180.0, 'OPENPOSITIONVALUE': 63622093152.0, 'VALUE': 2948223344.0, 'VOLUME': 37668, 'OPENPOSITION': 815208, 'SETTLEPRICE': 78044.0, 'SWAPRATE': 0.0, 'WAPRICE': 78289.0, 'CHANGE': -0.33, 'QTY': 1, 'NUMTRADES': 9506, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-20', 'SECID': 'SiU6', 'OPEN': 78029.0, 'LOW': 77180.0, 'HIGH': 78029.0, 'CLOSE': 77273.0, 'OPENPOSITIONVALUE': 69015782160.0, 'VALUE': 4104165148.0, 'VOLUME': 52795, 'OPENPOSITION': 891504, 'SETTLEPRICE': 77415.0, 'SWAPRATE': 0.0, 'WAPRICE': 77528.0, 'CHANGE': -1.16, 'QTY': 1, 'NUMTRADES': 12433, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-21', 'SECID': 'SiU6', 'OPEN': 77265.0, 'LOW': 76585.0, 'HIGH': 77388.0, 'CLOSE': 76889.0, 'OPENPOSITIONVALUE': 71473387332.0, 'VALUE': 4240063342.0, 'VOLUME': 55139, 'OPENPOSITION': 930316, 'SETTLEPRICE': 76827.0, 'SWAPRATE': 0.0, 'WAPRICE': 76816.0, 'CHANGE': -0.5, 'QTY': 1, 'NUMTRADES': 20852, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-22', 'SECID': 'SiU6', 'OPEN': 76889.0, 'LOW': 76762.0, 'HIGH': 77498.0, 'CLOSE': 77089.0, 'OPENPOSITIONVALUE': 78936845640.0, 'VALUE': 6097974469.0, 'VOLUME': 79283, 'OPENPOSITION': 1026140, 'SETTLEPRICE': 76926.0, 'SWAPRATE': 0.0, 'WAPRICE': 77064.0, 'CHANGE': 0.26, 'QTY': 1, 'NUMTRADES': 23071, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-23', 'SECID': 'SiU6', 'OPEN': 77052.0, 'LOW': 76539.0, 'HIGH': 78206.0, 'CLOSE': 78110.0, 'OPENPOSITIONVALUE': 91441575386.0, 'VALUE': 13276198958.0, 'VOLUME': 172258, 'OPENPOSITION': 1172794, 'SETTLEPRICE': 77969.0, 'SWAPRATE': 0.0, 'WAPRICE': 77106.0, 'CHANGE': 1.32, 'QTY': 10, 'NUMTRADES': 43942, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-24', 'SECID': 'SiU6', 'OPEN': 78160.0, 'LOW': 77150.0, 'HIGH': 78440.0, 'CLOSE': 77156.0, 'OPENPOSITIONVALUE': 97551738160.0, 'VALUE': 7689393834.0, 'VOLUME': 99080, 'OPENPOSITION': 1263820, 'SETTLEPRICE': 77188.0, 'SWAPRATE': 0.0, 'WAPRICE': 77599.0, 'CHANGE': -1.22, 'QTY': 2, 'NUMTRADES': 32004, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-27', 'SECID': 'SiU6', 'OPEN': 77499.0, 'LOW': 76500.0, 'HIGH': 77499.0, 'CLOSE': 76659.0, 'OPENPOSITIONVALUE': 103939544346.0, 'VALUE': 6717852508.0, 'VOLUME': 87530, 'OPENPOSITION': 1356258, 'SETTLEPRICE': 76637.0, 'SWAPRATE': 0.0, 'WAPRICE': 76723.0, 'CHANGE': -0.64, 'QTY': 4, 'NUMTRADES': 21428, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-28', 'SECID': 'SiU6', 'OPEN': 76518.0, 'LOW': 76229.0, 'HIGH': 77366.0, 'CLOSE': 77203.0, 'OPENPOSITIONVALUE': 109151251458.0, 'VALUE': 3877989759.0, 'VOLUME': 50484, 'OPENPOSITION': 1413858, 'SETTLEPRICE': 77201.0, 'SWAPRATE': 0.0, 'WAPRICE': 76977.0, 'CHANGE': 0.71, 'QTY': 1, 'NUMTRADES': 13390, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-29', 'SECID': 'SiU6', 'OPEN': 77215.0, 'LOW': 76579.0, 'HIGH': 77471.0, 'CLOSE': 76878.0, 'OPENPOSITIONVALUE': 111344545674.0, 'VALUE': 4391405209.0, 'VOLUME': 57059, 'OPENPOSITION': 1450422, 'SETTLEPRICE': 76767.0, 'SWAPRATE': 0.0, 'WAPRICE': 76861.0, 'CHANGE': -0.42, 'QTY': 1, 'NUMTRADES': 15890, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-04-30', 'SECID': 'SiU6', 'OPEN': 76755.0, 'LOW': 76551.0, 'HIGH': 77251.0, 'CLOSE': 77065.0, 'OPENPOSITIONVALUE': 116989046642.0, 'VALUE': 5605384671.0, 'VOLUME': 72973, 'OPENPOSITION': 1519358, 'SETTLEPRICE': 76999.0, 'SWAPRATE': 0.0, 'WAPRICE': 76849.0, 'CHANGE': 0.24, 'QTY': 2, 'NUMTRADES': 19368, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-04', 'SECID': 'SiU6', 'OPEN': 76710.0, 'LOW': 76710.0, 'HIGH': 77813.0, 'CLOSE': 77544.0, 'OPENPOSITIONVALUE': 121638557520.0, 'VALUE': 4483619843.0, 'VOLUME': 57995, 'OPENPOSITION': 1570320, 'SETTLEPRICE': 77461.0, 'SWAPRATE': 0.0, 'WAPRICE': 77467.0, 'CHANGE': 0.62, 'QTY': 1, 'NUMTRADES': 14585, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-05', 'SECID': 'SiU6', 'OPEN': 77533.0, 'LOW': 77322.0, 'HIGH': 77946.0, 'CLOSE': 77821.0, 'OPENPOSITIONVALUE': 123411006160.0, 'VALUE': 2764127119.0, 'VOLUME': 35629, 'OPENPOSITION': 1587484, 'SETTLEPRICE': 77740.0, 'SWAPRATE': 0.0, 'WAPRICE': 77611.0, 'CHANGE': 0.36, 'QTY': 5, 'NUMTRADES': 12224, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-06', 'SECID': 'SiU6', 'OPEN': 77795.0, 'LOW': 76169.0, 'HIGH': 77950.0, 'CLOSE': 76515.0, 'OPENPOSITIONVALUE': 124506406220.0, 'VALUE': 6226182342.0, 'VOLUME': 80925, 'OPENPOSITION': 1627322, 'SETTLEPRICE': 76510.0, 'SWAPRATE': 0.0, 'WAPRICE': 76865.0, 'CHANGE': -1.68, 'QTY': 4, 'NUMTRADES': 24076, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-07', 'SECID': 'SiU6', 'OPEN': 76500.0, 'LOW': 76273.0, 'HIGH': 76874.0, 'CLOSE': 76634.0, 'OPENPOSITIONVALUE': 125955367080.0, 'VALUE': 3780775705.0, 'VOLUME': 49354, 'OPENPOSITION': 1644648, 'SETTLEPRICE': 76585.0, 'SWAPRATE': 0.0, 'WAPRICE': 76622.0, 'CHANGE': 0.16, 'QTY': 4, 'NUMTRADES': 15444, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-08', 'SECID': 'SiU6', 'OPEN': 76600.0, 'LOW': 75919.0, 'HIGH': 76674.0, 'CLOSE': 76111.0, 'OPENPOSITIONVALUE': 126302372160.0, 'VALUE': 4332925401.0, 'VOLUME': 56779, 'OPENPOSITION': 1657120, 'SETTLEPRICE': 76218.0, 'SWAPRATE': 0.0, 'WAPRICE': 76266.0, 'CHANGE': -0.68, 'QTY': 1, 'NUMTRADES': 16678, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-11', 'SECID': 'SiU6', 'OPEN': 76051.0, 'LOW': 75221.0, 'HIGH': 76205.0, 'CLOSE': 75387.0, 'OPENPOSITIONVALUE': 124819539000.0, 'VALUE': 1922488179.0, 'VOLUME': 25465, 'OPENPOSITION': 1657630, 'SETTLEPRICE': 75300.0, 'SWAPRATE': 0.0, 'WAPRICE': 75486.0, 'CHANGE': -0.95, 'QTY': 1, 'NUMTRADES': 10559, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-12', 'SECID': 'SiU6', 'OPEN': 75450.0, 'LOW': 75264.0, 'HIGH': 76053.0, 'CLOSE': 75689.0, 'OPENPOSITIONVALUE': 130221145598.0, 'VALUE': 4719744852.0, 'VOLUME': 62352, 'OPENPOSITION': 1720022, 'SETTLEPRICE': 75709.0, 'SWAPRATE': 0.0, 'WAPRICE': 75711.0, 'CHANGE': 0.4, 'QTY': 1, 'NUMTRADES': 15385, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-13', 'SECID': 'SiU6', 'OPEN': 75839.0, 'LOW': 74865.0, 'HIGH': 75839.0, 'CLOSE': 75350.0, 'OPENPOSITIONVALUE': 142264700592.0, 'VALUE': 9934180635.0, 'VOLUME': 131688, 'OPENPOSITION': 1888152, 'SETTLEPRICE': 75346.0, 'SWAPRATE': 0.0, 'WAPRICE': 75270.0, 'CHANGE': -0.45, 'QTY': 1, 'NUMTRADES': 19977, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-14', 'SECID': 'SiU6', 'OPEN': 75388.0, 'LOW': 74761.0, 'HIGH': 75407.0, 'CLOSE': 75025.0, 'OPENPOSITIONVALUE': 147816487296.0, 'VALUE': 7174968524.0, 'VOLUME': 95549, 'OPENPOSITION': 1969416, 'SETTLEPRICE': 75056.0, 'SWAPRATE': 0.0, 'WAPRICE': 75052.0, 'CHANGE': -0.43, 'QTY': 1, 'NUMTRADES': 24465, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-15', 'SECID': 'SiU6', 'OPEN': 75056.0, 'LOW': 74605.0, 'HIGH': 75328.0, 'CLOSE': 74687.0, 'OPENPOSITIONVALUE': 153779164256.0, 'VALUE': 6494724081.0, 'VOLUME': 86758, 'OPENPOSITION': 2057852, 'SETTLEPRICE': 74728.0, 'SWAPRATE': 0.0, 'WAPRICE': 74796.0, 'CHANGE': -0.45, 'QTY': 2, 'NUMTRADES': 17498, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-18', 'SECID': 'SiU6', 'OPEN': 74721.0, 'LOW': 73691.0, 'HIGH': 74755.0, 'CLOSE': 73692.0, 'OPENPOSITIONVALUE': 160866449700.0, 'VALUE': 9814912072.0, 'VOLUME': 132177, 'OPENPOSITION': 2172990, 'SETTLEPRICE': 74030.0, 'SWAPRATE': 0.0, 'WAPRICE': 74072.0, 'CHANGE': -1.33, 'QTY': 2, 'NUMTRADES': 26020, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-19', 'SECID': 'SiU6', 'OPEN': 73700.0, 'LOW': 72045.0, 'HIGH': 73826.0, 'CLOSE': 72046.0, 'OPENPOSITIONVALUE': 164881294700.0, 'VALUE': 11036371313.0, 'VOLUME': 150969, 'OPENPOSITION': 2278310, 'SETTLEPRICE': 72370.0, 'SWAPRATE': 0.0, 'WAPRICE': 72867.0, 'CHANGE': -2.23, 'QTY': 1, 'NUMTRADES': 36243, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-20', 'SECID': 'SiU6', 'OPEN': 72046.0, 'LOW': 71566.0, 'HIGH': 73169.0, 'CLOSE': 72984.0, 'OPENPOSITIONVALUE': 175469581750.0, 'VALUE': 9899058374.0, 'VOLUME': 136330, 'OPENPOSITION': 2402870, 'SETTLEPRICE': 73025.0, 'SWAPRATE': 0.0, 'WAPRICE': 72709.0, 'CHANGE': 1.3, 'QTY': 1, 'NUMTRADES': 32951, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-21', 'SECID': 'SiU6', 'OPEN': 72902.0, 'LOW': 72184.0, 'HIGH': 73299.0, 'CLOSE': 72733.0, 'OPENPOSITIONVALUE': 188221308100.0, 'VALUE': 11545908936.0, 'VOLUME': 158593, 'OPENPOSITION': 2586950, 'SETTLEPRICE': 72758.0, 'SWAPRATE': 0.0, 'WAPRICE': 72600.0, 'CHANGE': -0.34, 'QTY': 1, 'NUMTRADES': 28734, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-22', 'SECID': 'SiU6', 'OPEN': 72770.0, 'LOW': 72583.0, 'HIGH': 73713.0, 'CLOSE': 73463.0, 'OPENPOSITIONVALUE': 205684923544.0, 'VALUE': 14518924493.0, 'VOLUME': 199056, 'OPENPOSITION': 2802476, 'SETTLEPRICE': 73394.0, 'SWAPRATE': 0.0, 'WAPRICE': 73048.0, 'CHANGE': 1.0, 'QTY': 1, 'NUMTRADES': 33521, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-25', 'SECID': 'SiU6', 'OPEN': 73763.0, 'LOW': 72982.0, 'HIGH': 74161.0, 'CLOSE': 74030.0, 'OPENPOSITIONVALUE': 215882043210.0, 'VALUE': 12045185517.0, 'VOLUME': 163960, 'OPENPOSITION': 2923290, 'SETTLEPRICE': 73849.0, 'SWAPRATE': 0.0, 'WAPRICE': 73486.0, 'CHANGE': 0.77, 'QTY': 1, 'NUMTRADES': 34481, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-26', 'SECID': 'SiU6', 'OPEN': 73881.0, 'LOW': 72765.0, 'HIGH': 74135.0, 'CLOSE': 72912.0, 'OPENPOSITIONVALUE': 226855939750.0, 'VALUE': 12062100472.0, 'VOLUME': 164310, 'OPENPOSITION': 3112946, 'SETTLEPRICE': 72875.0, 'SWAPRATE': 0.0, 'WAPRICE': 73313.0, 'CHANGE': -1.51, 'QTY': 1, 'NUMTRADES': 36820, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-27', 'SECID': 'SiU6', 'OPEN': 72912.0, 'LOW': 72419.0, 'HIGH': 73000.0, 'CLOSE': 72834.0, 'OPENPOSITIONVALUE': 247266269824.0, 'VALUE': 20556418797.0, 'VOLUME': 282537, 'OPENPOSITION': 3393904, 'SETTLEPRICE': 72856.0, 'SWAPRATE': 0.0, 'WAPRICE': 72729.0, 'CHANGE': -0.11, 'QTY': 4, 'NUMTRADES': 60973, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-28', 'SECID': 'SiU6', 'OPEN': 72840.0, 'LOW': 72340.0, 'HIGH': 73000.0, 'CLOSE': 72990.0, 'OPENPOSITIONVALUE': 263107650240.0, 'VALUE': 18180042774.0, 'VOLUME': 250310, 'OPENPOSITION': 3618092, 'SETTLEPRICE': 72720.0, 'SWAPRATE': 0.0, 'WAPRICE': 72606.0, 'CHANGE': 0.21, 'QTY': 5, 'NUMTRADES': 70927, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-05-29', 'SECID': 'SiU6', 'OPEN': 73030.0, 'LOW': 72303.0, 'HIGH': 73155.0, 'CLOSE': 72585.0, 'OPENPOSITIONVALUE': 290594478888.0, 'VALUE': 19169190089.0, 'VOLUME': 263972, 'OPENPOSITION': 4002348, 'SETTLEPRICE': 72606.0, 'SWAPRATE': 0.0, 'WAPRICE': 72564.0, 'CHANGE': -0.56, 'QTY': 5, 'NUMTRADES': 45468, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-01', 'SECID': 'SiU6', 'OPEN': 72488.0, 'LOW': 72488.0, 'HIGH': 73795.0, 'CLOSE': 73559.0, 'OPENPOSITIONVALUE': 315076736128.0, 'VALUE': 21384548379.0, 'VOLUME': 292660, 'OPENPOSITION': 4282796, 'SETTLEPRICE': 73568.0, 'SWAPRATE': 0.0, 'WAPRICE': 73332.0, 'CHANGE': 1.34, 'QTY': 1, 'NUMTRADES': 50122, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-02', 'SECID': 'SiU6', 'OPEN': 73568.0, 'LOW': 73422.0, 'HIGH': 74904.0, 'CLOSE': 74850.0, 'OPENPOSITIONVALUE': 350758341648.0, 'VALUE': 28428440635.0, 'VOLUME': 383955, 'OPENPOSITION': 4694052, 'SETTLEPRICE': 74724.0, 'SWAPRATE': 0.0, 'WAPRICE': 74335.0, 'CHANGE': 1.76, 'QTY': 1, 'NUMTRADES': 78056, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-03', 'SECID': 'SiU6', 'OPEN': 74900.0, 'LOW': 73934.0, 'HIGH': 75941.0, 'CLOSE': 75388.0, 'OPENPOSITIONVALUE': 373599126568.0, 'VALUE': 32082697003.0, 'VOLUME': 428014, 'OPENPOSITION': 4962728, 'SETTLEPRICE': 75281.0, 'SWAPRATE': 0.0, 'WAPRICE': 74987.0, 'CHANGE': 0.72, 'QTY': 1, 'NUMTRADES': 105141, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-04', 'SECID': 'SiU6', 'OPEN': 75408.0, 'LOW': 75026.0, 'HIGH': 76342.0, 'CLOSE': 75200.0, 'OPENPOSITIONVALUE': 403723238800.0, 'VALUE': 35120364159.0, 'VOLUME': 463652, 'OPENPOSITION': 5364380, 'SETTLEPRICE': 75260.0, 'SWAPRATE': 0.0, 'WAPRICE': 75906.0, 'CHANGE': -0.25, 'QTY': 2, 'NUMTRADES': 89196, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-05', 'SECID': 'SiU6', 'OPEN': 75295.0, 'LOW': 75100.0, 'HIGH': 75790.0, 'CLOSE': 75700.0, 'OPENPOSITIONVALUE': 422736933840.0, 'VALUE': 17288568953.0, 'VOLUME': 229375, 'OPENPOSITION': 5587324, 'SETTLEPRICE': 75660.0, 'SWAPRATE': 0.0, 'WAPRICE': 75484.0, 'CHANGE': 0.67, 'QTY': 1, 'NUMTRADES': 46491, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-08', 'SECID': 'SiU6', 'OPEN': 75732.0, 'LOW': 74430.0, 'HIGH': 75915.0, 'CLOSE': 74451.0, 'OPENPOSITIONVALUE': 454295707280.0, 'VALUE': 38834982185.0, 'VOLUME': 515941, 'OPENPOSITION': 6083720, 'SETTLEPRICE': 74674.0, 'SWAPRATE': 0.0, 'WAPRICE': 75082.0, 'CHANGE': -1.65, 'QTY': 2, 'NUMTRADES': 83464, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-09', 'SECID': 'SiU6', 'OPEN': 74451.0, 'LOW': 73250.0, 'HIGH': 74613.0, 'CLOSE': 73554.0, 'OPENPOSITIONVALUE': 473128522224.0, 'VALUE': 35714705759.0, 'VOLUME': 483459, 'OPENPOSITION': 6427678, 'SETTLEPRICE': 73608.0, 'SWAPRATE': 0.0, 'WAPRICE': 73684.0, 'CHANGE': -1.21, 'QTY': 1, 'NUMTRADES': 114901, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-10', 'SECID': 'SiU6', 'OPEN': 73540.0, 'LOW': 73064.0, 'HIGH': 74197.0, 'CLOSE': 74000.0, 'OPENPOSITIONVALUE': 503543830176.0, 'VALUE': 31674586467.0, 'VOLUME': 429232, 'OPENPOSITION': 6812748, 'SETTLEPRICE': 73912.0, 'SWAPRATE': 0.0, 'WAPRICE': 73818.0, 'CHANGE': 0.61, 'QTY': 1, 'NUMTRADES': 87462, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-11', 'SECID': 'SiU6', 'OPEN': 74018.0, 'LOW': 73587.0, 'HIGH': 74325.0, 'CLOSE': 74069.0, 'OPENPOSITIONVALUE': 529561940080.0, 'VALUE': 27576118809.0, 'VOLUME': 373005, 'OPENPOSITION': 7167866, 'SETTLEPRICE': 73880.0, 'SWAPRATE': 0.0, 'WAPRICE': 73913.0, 'CHANGE': 0.09, 'QTY': 2, 'NUMTRADES': 80832, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-15', 'SECID': 'SiU6', 'OPEN': 74490.0, 'LOW': 73544.0, 'HIGH': 75142.0, 'CLOSE': 73699.0, 'OPENPOSITIONVALUE': 647058975220.0, 'VALUE': 114004849081.0, 'VOLUME': 1540452, 'OPENPOSITION': 8776180, 'SETTLEPRICE': 73729.0, 'SWAPRATE': 0.0, 'WAPRICE': 74013.0, 'CHANGE': -0.5, 'QTY': 1, 'NUMTRADES': 259823, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-16', 'SECID': 'SiU6', 'OPEN': 73700.0, 'LOW': 73340.0, 'HIGH': 74597.0, 'CLOSE': 74508.0, 'OPENPOSITIONVALUE': 724209710900.0, 'VALUE': 82468991268.0, 'VOLUME': 1116249, 'OPENPOSITION': 9735310, 'SETTLEPRICE': 74390.0, 'SWAPRATE': 0.0, 'WAPRICE': 73932.0, 'CHANGE': 1.1, 'QTY': 1, 'NUMTRADES': 190954, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-17', 'SECID': 'SiU6', 'OPEN': 74514.0, 'LOW': 74275.0, 'HIGH': 75219.0, 'CLOSE': 75137.0, 'OPENPOSITIONVALUE': 782600390400.0, 'VALUE': 80644401912.0, 'VOLUME': 1079421, 'OPENPOSITION': 10440240, 'SETTLEPRICE': 74960.0, 'SWAPRATE': 0.0, 'WAPRICE': 74782.0, 'CHANGE': 0.84, 'QTY': 1, 'NUMTRADES': 196008, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-18', 'SECID': 'SiU6', 'OPEN': 75206.0, 'LOW': 74806.0, 'HIGH': 75500.0, 'CLOSE': 75292.0, 'OPENPOSITIONVALUE': 829707296940.0, 'VALUE': 98465702430.0, 'VOLUME': 1311002, 'OPENPOSITION': 11073690, 'SETTLEPRICE': 74926.0, 'SWAPRATE': 0.0, 'WAPRICE': 75143.0, 'CHANGE': 0.21, 'QTY': 3, 'NUMTRADES': 238778, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-19', 'SECID': 'SiU6', 'OPEN': 75320.0, 'LOW': 74811.0, 'HIGH': 75700.0, 'CLOSE': 75340.0, 'OPENPOSITIONVALUE': 835514455304.0, 'VALUE': 78408379763.0, 'VOLUME': 1042524, 'OPENPOSITION': 11115886, 'SETTLEPRICE': 75164.0, 'SWAPRATE': 0.0, 'WAPRICE': 75208.0, 'CHANGE': 0.06, 'QTY': 1, 'NUMTRADES': 235087, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-22', 'SECID': 'SiU6', 'OPEN': 75386.0, 'LOW': 74954.0, 'HIGH': 76629.0, 'CLOSE': 76551.0, 'OPENPOSITIONVALUE': 855770092316.0, 'VALUE': 98248706367.0, 'VOLUME': 1296959, 'OPENPOSITION': 11233822, 'SETTLEPRICE': 76178.0, 'SWAPRATE': 0.0, 'WAPRICE': 75753.0, 'CHANGE': 1.61, 'QTY': 1, 'NUMTRADES': 245508, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-23', 'SECID': 'SiU6', 'OPEN': 76560.0, 'LOW': 75704.0, 'HIGH': 76912.0, 'CLOSE': 75950.0, 'OPENPOSITIONVALUE': 867724398632.0, 'VALUE': 104075200535.0, 'VOLUME': 1366110, 'OPENPOSITION': 11438648, 'SETTLEPRICE': 75859.0, 'SWAPRATE': 0.0, 'WAPRICE': 76184.0, 'CHANGE': -0.79, 'QTY': 60, 'NUMTRADES': 243675, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-24', 'SECID': 'SiU6', 'OPEN': 76036.0, 'LOW': 75961.0, 'HIGH': 76580.0, 'CLOSE': 76373.0, 'OPENPOSITIONVALUE': 868982963298.0, 'VALUE': 84370765145.0, 'VOLUME': 1105712, 'OPENPOSITION': 11384106, 'SETTLEPRICE': 76333.0, 'SWAPRATE': 0.0, 'WAPRICE': 76305.0, 'CHANGE': 0.56, 'QTY': 11, 'NUMTRADES': 216215, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-25', 'SECID': 'SiU6', 'OPEN': 76370.0, 'LOW': 76281.0, 'HIGH': 78000.0, 'CLOSE': 77898.0, 'OPENPOSITIONVALUE': 884375327290.0, 'VALUE': 101661432068.0, 'VOLUME': 1317747, 'OPENPOSITION': 11358970, 'SETTLEPRICE': 77857.0, 'SWAPRATE': 0.0, 'WAPRICE': 77157.0, 'CHANGE': 2.0, 'QTY': 12, 'NUMTRADES': 278444, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-26', 'SECID': 'SiU6', 'OPEN': 78000.0, 'LOW': 77982.0, 'HIGH': 81000.0, 'CLOSE': 79981.0, 'OPENPOSITIONVALUE': 867779160960.0, 'VALUE': 233459044619.0, 'VOLUME': 2935892, 'OPENPOSITION': 10849952, 'SETTLEPRICE': 79980.0, 'SWAPRATE': 0.0, 'WAPRICE': 79538.0, 'CHANGE': 2.67, 'QTY': 1, 'NUMTRADES': 525020, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-29', 'SECID': 'SiU6', 'OPEN': 79988.0, 'LOW': 78206.0, 'HIGH': 80788.0, 'CLOSE': 79021.0, 'OPENPOSITIONVALUE': 868521743964.0, 'VALUE': 167770920733.0, 'VOLUME': 2110989, 'OPENPOSITION': 11013324, 'SETTLEPRICE': 78861.0, 'SWAPRATE': 0.0, 'WAPRICE': 79472.0, 'CHANGE': -1.2, 'QTY': 5, 'NUMTRADES': 424002, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}, {'BOARDID': 'RFUD', 'TRADEDATE': '2026-06-30', 'SECID': 'SiU6', 'OPEN': 79044.0, 'LOW': 78728.0, 'HIGH': 80908.0, 'CLOSE': 80865.0, 'OPENPOSITIONVALUE': 889145057680.0, 'VALUE': 133294652152.0, 'VOLUME': 1670171, 'OPENPOSITION': 11019272, 'SETTLEPRICE': 80690.0, 'SWAPRATE': 0.0, 'WAPRICE': 79810.0, 'CHANGE': 2.33, 'QTY': 5, 'NUMTRADES': 326226, 'SHORTNAME': 'Si-9.26', 'ASSETCODE': 'Si', 'SWAPRATE_CURR': None}]}
hist={s:{dt.date.fromisoformat(v['TRADEDATE']):v for v in rows} for s,rows in raw.items()}
FM=hist['SiM6'][start]['CLOSE']; FU=hist['SiU6'][start]['CLOSE']
km=math.log(FM/(1000*S0))/((em-start).days/365)
ku=math.log(FU/(1000*S0))/((eu-start).days/365)
Kfair=(FM+(FU-FM)*(end-em).days/(eu-em).days)/1000
Kbank=Kfair+.50 # bank credit/hedging spread assumption, RUB/USD
Bgrow=B*(1+r/365)**days
qndf=Bgrow/Kbank
cbr={dt.date.fromisoformat(k):v for k,v in {'2025-12-30': 77.4466, '2025-12-31': 78.2267, '2026-01-13': 78.7913, '2026-01-14': 78.8527, '2026-01-15': 78.5711, '2026-01-16': 78.528, '2026-01-17': 77.8332, '2026-01-20': 77.7586, '2026-01-21': 77.8247, '2026-01-22': 77.5159, '2026-01-23': 76.0382, '2026-01-24': 75.9246, '2026-01-27': 76.0101, '2026-01-28': 76.5519, '2026-01-29': 76.2662, '2026-01-30': 76.0251, '2026-01-31': 75.7327, '2026-02-03': 77.0223, '2026-02-04': 76.9817, '2026-02-05': 76.9102, '2026-02-06': 76.5523, '2026-02-07': 77.054, '2026-02-10': 77.6502, '2026-02-11': 77.2091, '2026-02-12': 77.4648, '2026-02-13': 77.188, '2026-02-14': 77.1944, '2026-02-17': 76.6201, '2026-02-18': 76.7389, '2026-02-19': 76.1524, '2026-02-20': 76.6405, '2026-02-21': 76.7519, '2026-02-25': 76.6342, '2026-02-26': 76.4678, '2026-02-27': 77.1218, '2026-02-28': 77.2736, '2026-03-03': 77.1734, '2026-03-04': 77.6093, '2026-03-05': 77.8009, '2026-03-06': 78.19, '2026-03-07': 79.15, '2026-03-11': 78.7396, '2026-03-12': 79.068, '2026-03-13': 79.0671, '2026-03-14': 80.2254, '2026-03-17': 81.0517, '2026-03-18': 81.9103, '2026-03-19': 83.1259, '2026-03-20': 84.8379, '2026-03-21': 83.9982, '2026-03-24': 81.8763, '2026-03-25': 80.9604, '2026-03-26': 80.7192, '2026-03-27': 82.1314, '2026-03-28': 81.1443, '2026-03-31': 81.2955, '2026-04-01': 81.2504, '2026-04-02': 80.6234, '2026-04-03': 80.3332, '2026-04-04': 79.7293, '2026-04-07': 78.7277, '2026-04-08': 78.7496, '2026-04-09': 78.3043, '2026-04-10': 77.8366, '2026-04-11': 76.9724, '2026-04-14': 76.2489, '2026-04-15': 75.8532, '2026-04-16': 75.2346, '2026-04-17': 76.0861, '2026-04-18': 76.0535, '2026-04-21': 75.237, '2026-04-22': 74.5897, '2026-04-23': 74.9995, '2026-04-24': 74.8349, '2026-04-25': 75.5273, '2026-04-28': 74.8081, '2026-04-29': 74.6947, '2026-04-30': 74.8806, '2026-05-01': 74.8014, '2026-05-05': 75.4388, '2026-05-06': 75.3448, '2026-05-07': 75.2246, '2026-05-08': 74.6209, '2026-05-09': 74.2963, '2026-05-13': 73.7878, '2026-05-14': 73.342, '2026-05-15': 73.1385, '2026-05-16': 73.1275, '2026-05-19': 72.3497, '2026-05-20': 71.2926, '2026-05-21': 70.9509, '2026-05-22': 70.7902, '2026-05-23': 71.209, '2026-05-26': 71.546, '2026-05-27': 71.668, '2026-05-28': 70.9012, '2026-05-29': 71.3715, '2026-05-30': 71.0224, '2026-06-02': 71.5532, '2026-06-03': 72.5597, '2026-06-04': 73.3436, '2026-06-05': 74.2956, '2026-06-06': 73.4689, '2026-06-09': 73.2644, '2026-06-10': 71.7318, '2026-06-11': 71.7892, '2026-06-12': 71.9077, '2026-06-16': 72.4513, '2026-06-17': 72.1388, '2026-06-18': 72.7479, '2026-06-19': 73.3591, '2026-06-20': 73.439, '2026-06-23': 73.765, '2026-06-24': 74.62, '2026-06-25': 74.7738, '2026-06-26': 75.6347, '2026-06-27': 77.0611, '2026-06-30': 77.7539}.items()}
ST=cbr[end]
def scenario(change,stress=False):
 spots=np.linspace(S0,S0*(1+change),days+1)
 if stress:
  spots=np.interp(np.arange(days+1),[0,30,130,days],[S0,.8*S0,.8*S0,S0*(1+change)])
 prices={}
 for s,k,e in [('SiM6',km,em),('SiU6',ku,eu)]:
  prices[s]=[1000*sp*math.exp(k*max((e-d).days,0)/365) for d,sp in zip(dates,spots)]
 return prices,spots
def actual_prices():
 prices={}
 for s in hist:
  last=hist[s][start]['CLOSE']; a=[]
  for d in dates:
   row=hist[s].get(d)
   if row:last=row['CLOSE'] if d in [start,roll,end] else row['SETTLEPRICE']
   a.append(last)
  prices[s]=a
 return prices
def simulate(n,prices,spot,margin=g):
 # Economic daily cash ledger; settlement timing proxy, no intraday broker cash ledger.
 G=n*FM*margin; cash=B-G-n*fee; cum=0.; interest=0.; maxdraw=0.; sumneg=0.; mincash=cash
 ledger=[]; prev=FM; s='SiM6'; ri=(roll-start).days
 for i,d in enumerate(dates[1:],1):
  ir=cash*r/365;cash+=ir;interest+=ir
  price=prices[s][i]; vm=n*(price-prev);cum+=vm;cash+=vm
  sumneg+=max(-vm,0);maxdraw=max(maxdraw,-cum); costs=0.
  if i==ri:
   costs=2*n*fee;s='SiU6';prev=prices[s][i]
  else:prev=price
  if d==end:costs+=n*fee
  cash-=costs;mincash=min(mincash,cash)
  ledger.append(dict(date=str(d),contract=s,vm=vm,cum_vm=cum,cash=cash,interest=ir,costs=costs))
 # One-day bank bridge at 20% for released margin and last positive VM (T+1).
 bridge_principal=G+max(ledger[-1]['vm'],0)
 bridge_cost=bridge_principal*.20/365
 wealth=cash+G-bridge_cost; usd=wealth/(spot*(1+basis))
 return dict(n=n,wealth=wealth,usd=usd,equiv=usd*S0,pnl=cum,interest=interest,fees=4*n*fee,bridge_cost=bridge_cost,bridge_principal=bridge_principal,GO=G,max_vm=maxdraw,sum_negative_vm=sumneg,min_free_cash=mincash,ledger=ledger)
# Size before seeing actual data: flatten USD outcomes across the two linear stress-free paths.
pLo,sLo=scenario(-.15);pHi,sHi=scenario(.15)
unitLo=simulate(1,pLo,sLo[-1])['wealth']-Bgrow
unitHi=simulate(1,pHi,sHi[-1])['wealth']-Bgrow
N=round(Bgrow*(1/sLo[-1]-1/sHi[-1])/(unitHi/sHi[-1]-unitLo/sLo[-1]))
Nhalf=round(N/2)
def ndf(spot):
 pnl=qndf*(spot-Kbank); wealth=Bgrow+pnl
 return dict(n=qndf,wealth=wealth,usd=wealth/(spot*(1+basis)),equiv=wealth/(spot*(1+basis))*S0,pnl=pnl,interest=Bgrow-B,fees=0,GO=0,max_vm=0,min_free_cash=B)
def plain(spot):return dict(wealth=Bgrow,usd=Bgrow/(spot*(1+basis)),equiv=Bgrow/(spot*(1+basis))*S0)
results={}; stresses={}
for c in [-.15,-.02,0,.02,.15]:
 prices,spots=scenario(c)
 results[str(c)]={'Full':simulate(N,prices,spots[-1]),'Half':simulate(Nhalf,prices,spots[-1]),'NDF':ndf(spots[-1]),'None':plain(spots[-1])}
 prices,spots=scenario(c,True)
 stresses[str(c)]={'Full':simulate(N,prices,spots[-1]),'Half':simulate(Nhalf,prices,spots[-1]),'NDF':ndf(spots[-1])}
ap=actual_prices(); actual={'Full':simulate(N,ap,ST),'Half':simulate(Nhalf,ap,ST),'NDF':ndf(ST),'None':plain(ST)}
sens=[]
for rate in [.10,.14,.18]:
 old=r;r=rate
 for label,n in [('Full',N),('Half',Nhalf)]:
  v=simulate(n,ap,ST);sens.append(dict(parameter='placement_rate',value=rate,structure=label,wealth=v['wealth'],usd=v['usd']))
 r=old
for spread in [0,.005,.01]:
 for label,v in actual.items():sens.append(dict(parameter='conversion_spread',value=spread,structure=label,usd=v['wealth']/(ST*(1+spread))))
for bump in [-1000,1000]:
 cp={s:list(v) for s,v in ap.items()};cp['SiU6'][-1]+=bump
 v=simulate(N,cp,ST);sens.append(dict(parameter='terminal_basis_RUB_per_USD',value=bump/1000,structure='Full',wealth=v['wealth'],usd=v['usd']))
def clean(v):
 if isinstance(v,dict):return {k:clean(x) for k,x in v.items() if k!='ledger'}
 if isinstance(v,list):return [clean(x) for x in v]
 return v
summary=dict(inputs=dict(B=B,S0=S0,ST=ST,FM=FM,FU=FU,r=r,g=g,fee=fee,basis=basis,days=days,Kfair=Kfair,Kbank=Kbank,N=N,Nhalf=Nhalf,qndf=qndf,Bgrow=Bgrow),scenarios=clean(results),stress=clean(stresses),actual=clean(actual),sensitivity=sens)


## Сводные расчёты
Размер полного хеджа определяется равенством долларовых результатов двух линейных сценариев, затем округляется до целых лотов; 50% — половина этого размера. Небольшое расхождение результатов возникает от округления и стоимости последнего моста. Керри уже включено в F: отдельно оно не вычитается. Ролл — две сделки; комиссии со всех четырёх сторон включены.

In [2]:
print('Полный хедж:', N, 'контрактов; половинный:',Nhalf)
print('NDF fair/bank:',round(Kfair,4),round(Kbank,4),'номинал USD:',round(qndf,2))
for sc, group in {**results, "H1 actual":actual}.items():
 print('\nСценарий:',sc)
 for name,v in group.items():
  print(name, 'W млн RUB =',round(v['wealth']/1e6,4),'U USD =',round(v['usd'],2), 'U*S0 млн =',round(v['equiv']/1e6,4))


Полный хедж: 1232 контрактов; половинный: 616
NDF fair/bank: 83.2127 83.7127 номинал USD: 1280914.85

Сценарий: -0.15
Full W млн RUB = 84.6991 U USD = 1267473.08 U*S0 млн = 99.1502
Half W млн RUB = 95.964 U USD = 1436045.27 U*S0 млн = 112.3371
NDF W млн RUB = 85.1715 U USD = 1274542.13 U*S0 млн = 99.7032
None W млн RUB = 107.2288 U USD = 1604617.46 U*S0 млн = 125.5239

Сценарий: -0.02
Full W млн RUB = 97.6566 U USD = 1267519.53 U*S0 млн = 99.1539
Half W млн RUB = 102.4427 U USD = 1329639.79 U*S0 млн = 104.0133
NDF W млн RUB = 98.1977 U USD = 1274542.13 U*S0 млн = 99.7032
None W млн RUB = 107.2288 U USD = 1391760.04 U*S0 млн = 108.8728

Сценарий: 0
Full W млн RUB = 99.6501 U USD = 1267525.61 U*S0 млн = 99.1543
Half W млн RUB = 103.4395 U USD = 1315725.22 U*S0 млн = 102.9248
NDF W млн RUB = 100.2017 U USD = 1274542.13 U*S0 млн = 99.7032
None W млн RUB = 107.2288 U USD = 1363924.84 U*S0 млн = 106.6953

Сценарий: 0.02
Full W млн RUB = 101.6436 U USD = 1267531.44 U*S0 млн = 99.1548
Half W м

## Ликвидность, −10% за месяц и стресс −20%
max_vm=max(−накопленная ВМ,0) — дефицит относительно старта. sum_negative_vm — оборот отрицательных начислений, не денежный резерв. Стресс не является абсолютной границей потерь. NDF требует банковского лимита: отсутствие ВМ справедливо только для договора без промежуточного залога.

In [3]:
for name,n in [('Full',N),('Half',Nhalf)]:
 v=stresses['0'][name]
 shock=n*(FM-.9*S0*1000*math.exp(km*((em-start).days-30)/365))
 print(name,'ВМ stress:',round(v['max_vm'],2),'ГО:',round(v['GO'],2),'ГО25%+ВМ:',round(n*FM*.25+v['max_vm'],2),'shock10:',round(shock,2))
 print('H1 max deficit:',round(actual[name]['max_vm'],2),'total negative VM:',round(actual[name]['sum_negative_vm'],2),'bridge:',round(actual[name]['bridge_cost'],2))
print('NDF stress exposure proxy:',round(qndf*(Kbank-.8*S0),2))


Full ВМ stress: 23962616.99 ГО: 15318072.0 ГО25%+ВМ: 49492736.99 shock10: 11146409.27
H1 max deficit: 14256704.0 total negative VM: 48142864.0 bridge: 9746.3
Half ВМ stress: 11981308.49 ГО: 7659036.0 ГО25%+ВМ: 24746368.49 shock10: 5573204.63
H1 max deficit: 7128352.0 total negative VM: 24071432.0 bridge: 4873.15
NDF stress exposure proxy: 27067423.18


## Чувствительность при неизменном размере сделки
Размещение 10/14/18%, конвертация 0/0,5/1%, выходной базис ±1 ₽/$. NDF ниже сохраняет исходный номинал Q, поэтому изменение реальной доходности нарушает точную фиксацию бюджета.

In [4]:
for row in sens: print(row)
for realized_rate in [.10,.14,.18]:
 A_real=B*(1+realized_rate/365)**days
 W_real=A_real+qndf*(ST-Kbank)
 print('NDF rate',realized_rate,'W',W_real,'USD',W_real/(ST*(1+basis)))
for cost in [20,100]:
 # incremental trading costs at four actual transaction dates, including foregone interest
 extra=N*(cost-fee)*((1+r/365)**days+2*(1+r/365)**((end-roll).days)+1)
 print('Full cost per side',cost,'wealth',actual['Full']['wealth']-extra)
print('Full initial participation %',100*N/hist['SiM6'][start]['VOLUME'])


{'parameter': 'placement_rate', 'value': 0.1, 'structure': 'Full', 'wealth': 99059295.91848196, 'usd': 1267672.2788243364}
{'parameter': 'placement_rate', 'value': 0.1, 'structure': 'Half', 'wealth': 102085643.86222303, 'usd': 1306400.7732961187}
{'parameter': 'placement_rate', 'value': 0.14, 'structure': 'Full', 'wealth': 100722042.0680265, 'usd': 1288950.6170252669}
{'parameter': 'placement_rate', 'value': 0.14, 'structure': 'Half', 'wealth': 103975429.15335481, 'usd': 1330584.5553862837}
{'parameter': 'placement_rate', 'value': 0.18, 'structure': 'Full', 'wealth': 102418944.7353319, 'usd': 1310666.0597937712}
{'parameter': 'placement_rate', 'value': 0.18, 'structure': 'Half', 'wealth': 105903488.34814845, 'usd': 1355258.1326665382}
{'parameter': 'conversion_spread', 'value': 0, 'structure': 'Full', 'usd': 1295395.370110393}
{'parameter': 'conversion_spread', 'value': 0, 'structure': 'Half', 'usd': 1337237.478163215}
{'parameter': 'conversion_spread', 'value': 0, 'structure': 'NDF', 

## Журнал и проверка
Журнал доступен как actual["Full"]["ledger"], экспорт ниже необязателен. SETTLEPRICE используется между сделками, CLOSE — в даты исполнения; это оценка результата по дневным ценам, не подтверждение исполнимости всех контрактов по последней сделке. Последняя расчётная цена и последняя сделка могут заметно различаться.

In [5]:
assert len(raw['SiM6']) == 116 and len(raw['SiU6']) == 124
assert hist['SiM6'][start]['CLOSE']==82890 and hist['SiU6'][end]['CLOSE']==80865
assert abs(actual['Full']['pnl']-N*((hist['SiM6'][roll]['CLOSE']-FM)+(hist['SiU6'][end]['CLOSE']-hist['SiU6'][roll]['CLOSE'])))<.01
for k in ['Full','Half']:
 v=actual[k]
 assert abs(v['wealth']-(B+v['pnl']+v['interest']-v['fees']-v['bridge_cost']))<.01
assert max(v['NDF']['usd'] for v in results.values())-min(v['NDF']['usd'] for v in results.values())<1e-6
assert all(v['min_free_cash']>0 for group in stresses.values() for v in group.values())
assert abs(actual['Half']['pnl']*2-actual['Full']['pnl'])<.01
assert (end-start).days==182 and start < roll < em < end < eu
print('Проверены: размеры рядов, даты, P&L, баланс денег, NDF, положительная ликвидность, масштабирование 50%.')
print('Пик оттока:',min(actual['Full']['ledger'],key=lambda x:x['cum_vm']))


Проверены: размеры рядов, даты, P&L, баланс денег, NDF, положительная ликвидность, масштабирование 50%.
Пик оттока: {'date': '2026-05-19', 'contract': 'SiM6', 'vm': -2029104.0, 'cum_vm': -14256704.0, 'cash': 74820056.18790422, 'interest': 29465.08853126634, 'costs': 0.0}


In [6]:
# Необязательный экспорт журнала после выполнения Run All:
# import csv
# with open('full_hedge_ledger.csv','w',newline='',encoding='utf-8-sig') as f:
#     writer=csv.DictWriter(f,fieldnames=actual['Full']['ledger'][0].keys())
#     writer.writeheader(); writer.writerows(actual['Full']['ledger'])
